In [1]:
# Install packages
!pip -q install geopandas pyogrio beautifulsoup4

In [2]:
# Import tools
import csv
import io
import json
import re
import shutil
import unicodedata
import zipfile

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

from bs4 import BeautifulSoup
from google.colab import files
from IPython.display import display
from itertools import product
from math import comb
from pathlib import Path
from scipy.linalg import qr
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics.pairwise import haversine_distances
from sklearn.pipeline import Pipeline
from urllib.parse import quote

In [3]:
# Set plot style
plt.rc("text",usetex=False)
plt.rc("font",family="serif")
plt.rc("mathtext",fontset="cm")
plt.rc("font",size=13)
plt.rc("axes",labelsize=14)
plt.rc("xtick",labelsize=11)
plt.rc("ytick",labelsize=11)
plt.rc("legend",fontsize=11)
plt.rc("xtick",top=False,direction="out")
plt.rc("ytick",right=False,direction="out")
plt.rc("xtick.major",size=4.5,width=0.9)
plt.rc("ytick.major",size=4.5,width=0.9)
plt.rc("lines",linewidth=1.8,markersize=4.5,markeredgewidth=0.8)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
colors = {"blue":"#0072B2","orange":"#D55E00","green":"#009E73","purple":"#CC79A7","gray":"#7F7F7F"}

In [4]:
# Set transfer protocol
RUN_STARTED_UTC = pd.Timestamp.now(tz="UTC").isoformat()
ACCESS_DATE_UTC = pd.Timestamp(RUN_STARTED_UTC).date().isoformat()
QUALIFICATION_START_YEAR = 2018
QUALIFICATION_END_YEAR = 2022
MIN_QUALIFICATION_YEARS = 4
MIN_SEASON_DAYS = 40
VALIDATION_YEAR = 2023
TEST_YEAR = 2024
TARGET_COVERAGE_THRESHOLD = .85
GRAPH_PRUNE_RATIO = 1e-6
RANDOM_SEED = 7
RANDOM_SETS = 200
BOOTSTRAP_REPLICATES = 1000
SENSOR_BUDGET = 10
EXACT_TOLERANCE = 1e-6
OBSERVATION_OFFSETS = [-4,-3,-2]

In [5]:
# Define EPA audit tools
def epa_sitekey(data):
    state = data["state"].astype(int).astype(str).str.zfill(2)
    county = data["county"].astype(int).astype(str).str.zfill(3)
    site = data["number"].astype(int).astype(str).str.zfill(4)
    return state+"-"+county+"-"+site

def epa_audit(state_code,region):
    frames = []
    for year in range(2018,2025):
        data = pd.read_csv(
            f"https://aqs.epa.gov/aqsweb/airdata/daily_88101_{year}.zip",
            compression="zip",
            low_memory=False
        )
        data = data.rename(columns={
            "State Code":"state",
            "County Code":"county",
            "Site Num":"number",
            "POC":"poc",
            "Date Local":"date",
            "Arithmetic Mean":"pm",
            "Latitude":"lat",
            "Longitude":"lon",
            "Sample Duration":"duration",
            "Pollutant Standard":"standard",
            "Event Type":"event",
            "Observation Percent":"observed"
        })
        data = data[pd.to_numeric(data["state"],errors="coerce").eq(state_code)].copy()
        data["date"] = pd.to_datetime(data["date"])
        data["pm"] = pd.to_numeric(data["pm"],errors="coerce")
        data["observed"] = pd.to_numeric(data["observed"],errors="coerce")
        data["site"] = epa_sitekey(data)
        frames.append(data[data["date"].dt.month.between(6,10)])
    raw = pd.concat(frames,ignore_index=True)
    raw["event"] = raw["event"].fillna("No Events")
    selected = raw[
        raw["standard"].eq("PM25 24-hour 2012")
        &raw["event"].isin(["No Events","Included"])
        &raw["duration"].isin(["24-HR BLK AVG","24 HOUR"])
    ].copy()
    selected["eventrank"] = selected["event"].eq("Included").astype(int)
    selected["durationrank"] = selected["duration"].map({"24-HR BLK AVG":0,"24 HOUR":1})
    selected = selected.sort_values(
        ["site","poc","date","duration","eventrank","observed"],
        ascending=[True,True,True,True,False,False]
    ).drop_duplicates(["site","poc","date","duration"])
    selected = selected.sort_values(["site","poc","date","durationrank"]).drop_duplicates(["site","poc","date"])
    selected = selected[selected["duration"].eq("24 HOUR")|selected["observed"].ge(75)]
    daily = selected.groupby(["site","date"],as_index=False).agg(
        pm=("pm","mean"),
        lat=("lat","median"),
        lon=("lon","median"),
        monitors=("poc","nunique")
    )
    daily["year"] = daily["date"].dt.year
    coverage = daily.groupby(["site","year"],as_index=False).agg(days=("date","nunique"))
    stats = coverage.groupby("site",as_index=False).agg(
        years=("year","nunique"),
        days=("days","median"),
        total=("days","sum")
    )
    stats["complete"] = stats["total"]/(7*153)
    stats["qualified"] = stats["years"].ge(5)&stats["days"].ge(40)
    qualified = stats.loc[stats["qualified"],"site"]
    years = coverage[coverage["site"].isin(qualified)].groupby("year",as_index=False).agg(
        qualified_sites=("site","nunique"),
        median_days=("days","median")
    )
    poc = selected.groupby(["site","date"],as_index=False).agg(
        monitors=("poc","nunique"),
        minimum=("pm","min"),
        maximum=("pm","max")
    )
    poc["range"] = poc["maximum"]-poc["minimum"]
    shifts = []
    for site,data in daily.groupby("site"):
        lat0 = np.radians(data["lat"].median())
        lon0 = np.radians(data["lon"].median())
        lat = np.radians(data["lat"].to_numpy())
        lon = np.radians(data["lon"].to_numpy())
        a = np.sin((lat-lat0)/2)**2+np.cos(lat0)*np.cos(lat)*np.sin((lon-lon0)/2)**2
        shift = 2*6371.0088*np.arcsin(np.sqrt(np.clip(a,0,1)))
        shifts.append({"site":site,"max_shift_km":float(shift.max())})
    coords = pd.DataFrame(shifts)
    summary = {
        "region":region,
        "country":"United States",
        "raw_sites":len(stats),
        "qualified_sites":int(stats["qualified"].sum()),
        "median_days":stats["days"].median(),
        "median_completeness":stats["complete"].median(),
        "qualification_rate":stats["qualified"].mean(),
        "source":"EPA AirData",
        "decision":"screened out",
        "reproduced_here":True
    }
    return {"daily":daily,"stats":stats,"years":years,"poc":poc,"coords":coords,"summary":summary}

In [6]:
# Audit Oregon and Washington
us_audits = {
    "Oregon":epa_audit(41,"Oregon"),
    "Washington":epa_audit(53,"Washington")
}
us_summary = pd.DataFrame([us_audits[name]["summary"] for name in ["Washington","Oregon"]])
checks = {
    "Oregon qualified sites":int(us_summary.loc[us_summary["region"].eq("Oregon"),"qualified_sites"].iloc[0])==12,
    "Washington qualified sites":int(us_summary.loc[us_summary["region"].eq("Washington"),"qualified_sites"].iloc[0])==17
}
usauditchecks = pd.DataFrame({"check":checks.keys(),"passed":checks.values()})
display(us_summary.round(4))
display(usauditchecks)
if not usauditchecks["passed"].all():
    failed = usauditchecks.loc[~usauditchecks["passed"],"check"].tolist()
    raise ValueError("Failed U.S. transfer audit: "+", ".join(failed))

,region,country,raw_sites,qualified_sites,median_days,median_completeness,qualification_rate,source,decision,reproduced_here
0,Washington,United States,26,17,148.5,0.9108,0.6538,EPA AirData,screened out,True
1,Oregon,United States,18,12,104.5,0.4958,0.6667,EPA AirData,screened out,True


,check,passed
0,Oregon qualified sites,True
1,Washington qualified sites,True


In [7]:
# Define UK-AIR tools
def ukkey(value):
    text = unicodedata.normalize("NFKD",str(value)).encode("ascii","ignore").decode()
    return re.sub(r"[^a-z0-9]","",text.lower())

def ukair_table(content):
    try:
        text = content.decode("utf-8-sig")
    except UnicodeDecodeError:
        text = content.decode("cp1252")
    lines = text.splitlines()
    header = None
    for index,line in enumerate(lines):
        try:
            values = next(csv.reader([line]))
        except csv.Error:
            continue
        if len(values)>=20 and any("date" in str(value).strip().lower() for value in values):
            header = index
            break
    if header is None:
        raise ValueError("Could not identify UK-AIR header")
    data = pd.read_csv(io.StringIO("\n".join(lines[header:])),low_memory=False)
    datecols = [column for column in data.columns if "date" in str(column).lower()]
    if len(datecols)!=1:
        raise ValueError(f"Expected one date column, found {datecols}")
    hours = [column for column in data.columns if column!=datecols[0]]
    if len(hours)!=24:
        raise ValueError(f"Expected 24 hourly columns, found {len(hours)}")
    return data,datecols[0],hours

In [8]:
# Build England AURN metadata
currenturl = "https://uk-air.defra.gov.uk/networks/find-sites?action=results&country_id=9999&group_id=4&location_type=9999&pollutant=&region_id=9999&site_name=&view=advanced"
closedurl = "https://uk-air.defra.gov.uk/networks/aurn-sites"
session = requests.Session()
current = session.get(currenturl,timeout=120)
closed = session.get(closedurl,timeout=120)
current.raise_for_status()
closed.raise_for_status()
currentids = sorted(set(re.findall(r"UKA\d{5}",current.text)))
soup = BeautifulSoup(closed.text,"html.parser")
closedcodes = []
for link in soup.find_all("a",href=True):
    match = re.search(r"(?:aurn-)?site-info\?site_id=([^&#\"']+)",link["href"])
    if match:
        closedcodes.append(match.group(1))
closedcodes = sorted(set(closedcodes))
currentrefs = pd.DataFrame({
    "kind":"current",
    "key":currentids,
    "url":[f"https://uk-air.defra.gov.uk/networks/site-info?provider=&uka_id={value}" for value in currentids]
})
closedrefs = pd.DataFrame({
    "kind":"closed",
    "key":closedcodes,
    "url":[f"https://uk-air.defra.gov.uk/networks/site-info?site_id={value}" for value in closedcodes]
})
refs = pd.concat([currentrefs,closedrefs],ignore_index=True)
rows = []
for number,row in enumerate(refs.itertuples(),1):
    if number==1 or number%50==0:
        print("Reading AURN site",number,"of",len(refs))
    response = session.get(row.url,timeout=60)
    if not response.ok:
        continue
    page = BeautifulSoup(response.text,"html.parser")
    text = page.get_text("\n",strip=True)
    uka = re.search(r"UK-AIR ID:\s*(UKA\d{5})",text)
    region = re.search(r"Government Region:\s*([^\n]+)",text)
    coords = re.search(r"Latitude/Longitude:\s*(-?\d+(?:\.\d+)?)\s*,\s*(-?\d+(?:\.\d+)?)",text)
    flat = None
    for link in page.find_all("a",href=True):
        match = re.search(r"flat_files\?site_id=([^&#\"']+)",link["href"])
        if match:
            flat = match.group(1)
            break
    start = np.nan
    end = np.nan
    for table_row in page.find_all("tr"):
        values = [cell.get_text(" ",strip=True) for cell in table_row.find_all(["th","td"])]
        pollutant = ukkey(values[0]) if values else ""
        if "pm25particulatematterhourlymeasured" in pollutant and "nonvolatile" not in pollutant and not pollutant.startswith("volatile"):
            if len(values)>=3:
                start,end = values[1],values[2]
            break
    rows.append({
        "uka":uka.group(1) if uka else np.nan,
        "code":flat,
        "region":region.group(1).strip() if region else np.nan,
        "lat":float(coords.group(1)) if coords else np.nan,
        "lon":float(coords.group(2)) if coords else np.nan,
        "pm25_start":start,
        "pm25_end":end
    })
ukmeta = pd.DataFrame(rows)
ukmeta["start"] = pd.to_datetime(ukmeta["pm25_start"],errors="coerce",dayfirst=True)
ukmeta["end"] = pd.to_datetime(ukmeta["pm25_end"].replace({"-":np.nan,"":np.nan}),errors="coerce",dayfirst=True)
ukmeta["pm_overlap"] = (
    ukmeta["start"].notna()
    &ukmeta["start"].le(pd.Timestamp("2024-10-31"))
    &(ukmeta["end"].isna()|ukmeta["end"].ge(pd.Timestamp("2018-06-01")))
)
devolved = ukmeta["region"].astype(str).str.contains("Scotland|Wales|Northern Ireland",case=False,regex=True,na=False)
englandmeta = ukmeta[
    ukmeta["pm_overlap"]
    &~devolved
    &ukmeta["region"].notna()
    &ukmeta["uka"].notna()
    &ukmeta["code"].notna()
    &ukmeta["lat"].notna()
    &ukmeta["lon"].notna()
].drop_duplicates("uka").copy()
print("Historical references:",len(refs))
print("England PM2.5 candidates:",len(englandmeta))

Reading AURN site 1 of 325
Reading AURN site 50 of 325
Reading AURN site 100 of 325
Reading AURN site 150 of 325
Reading AURN site 200 of 325
Reading AURN site 250 of 325
Reading AURN site 300 of 325
Historical references: 325
England PM2.5 candidates: 91


In [9]:
# Download England PM2.5 data
englandrows = []
englandfilesrows = []
for number,row in enumerate(englandmeta.itertuples(),1):
    if number==1 or number%20==0:
        print("Downloading England site",number,"of",len(englandmeta))
    for year in range(2018,2025):
        if row.start>pd.Timestamp(f"{year}-10-31") or (pd.notna(row.end) and row.end<pd.Timestamp(f"{year}-06-01")):
            continue
        url = f"https://uk-air.defra.gov.uk/datastore/data_files/site_pol_data/{row.code}_PM25_{year}.csv"
        response = session.get(url,timeout=60)
        kind = response.headers.get("content-type","").lower()
        if not response.ok or "html" in kind:
            englandfilesrows.append({"site":row.uka,"year":year,"status":"missing"})
            continue
        try:
            data,datecol,hours = ukair_table(response.content)
            dates = pd.to_datetime(data[datecol],errors="coerce",dayfirst=True)
            values = data[hours].apply(pd.to_numeric,errors="coerce")
            daily = pd.DataFrame({
                "site":row.uka,
                "date":dates,
                "valid_hours":values.notna().sum(axis=1),
                "pm":values.mean(axis=1),
                "lat":row.lat,
                "lon":row.lon
            })
            daily = daily[daily["date"].dt.month.between(6,10)&daily["valid_hours"].ge(18)].copy()
            daily["year"] = year
            englandrows.append(daily)
            englandfilesrows.append({"site":row.uka,"year":year,"status":"ok","days":len(daily)})
        except Exception as error:
            englandfilesrows.append({"site":row.uka,"year":year,"status":"parse_error","detail":str(error)})
if not englandrows:
    raise RuntimeError("No England PM2.5 files parsed")
englanddaily = pd.concat(englandrows,ignore_index=True)
englandfiles = pd.DataFrame(englandfilesrows)

/tmp/ipykernel_2255/3780157200.py:18: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dates = pd.to_datetime(data[datecol],errors="coerce",dayfirst=True)


In [10]:
# Audit England PM2.5 coverage
englandcoverage = englanddaily.groupby(["site","year"],as_index=False).agg(days=("date","nunique"))
englandstats = englandcoverage.groupby("site",as_index=False).agg(
    years=("year","nunique"),
    days=("days","median"),
    total=("days","sum")
)
englandstats["complete"] = englandstats["total"]/(7*153)
englandstats["qualified"] = englandstats["years"].ge(5)&englandstats["days"].ge(40)
qualified = englandstats.loc[englandstats["qualified"],"site"]
englandyears = englandcoverage[englandcoverage["site"].isin(qualified)].groupby("year",as_index=False).agg(
    qualified_sites=("site","nunique"),
    median_days=("days","median")
)
england_summary = {
    "region":"England",
    "country":"United Kingdom",
    "raw_sites":len(englandstats),
    "qualified_sites":int(englandstats["qualified"].sum()),
    "median_days":englandstats["days"].median(),
    "median_completeness":englandstats["complete"].median(),
    "qualification_rate":englandstats["qualified"].mean(),
    "source":"UK-AIR AURN",
    "decision":"feasible external audit",
    "reproduced_here":True
}
checks = {
    "England raw sites":len(englandstats)==87,
    "England qualified sites":int(englandstats["qualified"].sum())==53,
    "No missing qualification years":englandyears["year"].nunique()==7
}
englandchecks = pd.DataFrame({"check":checks.keys(),"passed":checks.values()})
display(pd.DataFrame([england_summary]).round(4))
display(englandyears.round(2))
display(englandchecks)
if not englandchecks["passed"].all():
    failed = englandchecks.loc[~englandchecks["passed"],"check"].tolist()
    raise ValueError("Failed England audit: "+", ".join(failed))

,region,country,raw_sites,qualified_sites,median_days,median_completeness,qualification_rate,source,decision,reproduced_here
0,England,United Kingdom,87,53,151.0,0.8413,0.6092,UK-AIR AURN,feasible external audit,True


,year,qualified_sites,median_days
0,2018,52,141.5
1,2019,52,146.5
2,2020,52,150.0
3,2021,52,150.5
4,2022,53,149.0
5,2023,53,149.0
6,2024,51,151.0


,check,passed
0,England raw sites,True
1,England qualified sites,True
2,No missing qualification years,True


In [11]:
# Define NAPS download tools
def naps_csv(path):
    path = "/"+path.lstrip("/")
    url = "https://data-donnees.az.ec.gc.ca/api/file?path="+quote(path,safe="")
    response = requests.get(url,timeout=120)
    response.raise_for_status()
    try:
        text = response.content.decode("utf-8-sig")
    except UnicodeDecodeError:
        text = response.content.decode("cp1252")
    lines = text.splitlines()
    widths = []
    for line in lines:
        if not line.strip():
            widths.append(0)
            continue
        try:
            widths.append(len(next(csv.reader([line]))))
        except csv.Error:
            widths.append(0)
    header = None
    for index,width in enumerate(widths):
        if width<3:
            continue
        following = [value for value in widths[index+1:index+6] if value>0]
        if following and sum(value==width for value in following)>=min(3,len(following)):
            header = index
            break
    if header is None:
        raise ValueError("Could not identify NAPS table header")
    data = pd.read_csv(io.StringIO("\n".join(lines[header:])),low_memory=False)
    data.attrs["header_row"] = header+1
    return data,url

def colkey(name):
    text = unicodedata.normalize("NFKD",str(name)).encode("ascii","ignore").decode()
    return "".join(char.lower() for char in text if char.isalnum())

def findcol(data,prefix):
    found = [column for column in data.columns if colkey(column).startswith(prefix)]
    if len(found)!=1:
        raise ValueError(f"Expected one {prefix} column, found {found}")
    return found[0]

In [12]:
# Download NAPS PM2.5 data
root = "air/monitor/national-air-pollution-surveillance-naps-program"
naps = {}
downloadrows = []
for year in range(2018,2025):
    print("Loading",year)
    path = root+f"/Data-Donnees/{year}/ContinuousData-DonneesContinu/HourlyData-DonneesHoraires/PM25_{year}.csv"
    data,url = naps_csv(path)
    naps[year] = data
    downloadrows.append({
        "year":year,
        "rows":len(data),
        "columns":len(data.columns),
        "header_row":data.attrs["header_row"]
    })
naps_download_audit = pd.DataFrame(downloadrows)
if not all(list(naps[year].columns)==list(naps[2024].columns) for year in range(2018,2025)):
    raise ValueError("NAPS schema changed across years")
display(naps_download_audit)

Loading 2018
Loading 2019
Loading 2020
Loading 2021
Loading 2022
Loading 2023
Loading 2024


,year,rows,columns,header_row
0,2018,90885,32,8
1,2019,97455,32,8
2,2020,92964,32,8
3,2021,85410,32,8
4,2022,98185,32,8
5,2023,110595,32,8
6,2024,78324,32,8


In [13]:
# Build NAPS station-day data
dailyrows = []
sentinelrows = []
for year,data in naps.items():
    sitecol = findcol(data,"napsid")
    methodcol = findcol(data,"methodcode")
    provincecol = findcol(data,"provinceterritory")
    latcol = findcol(data,"latitude")
    loncol = findcol(data,"longitude")
    datecol = findcol(data,"date")
    hourcols = [column for column in data.columns if str(column).split("//")[0] in [f"H{hour:02d}" for hour in range(1,25)]]
    if len(hourcols)!=24:
        raise ValueError(f"{year}: expected 24 hourly columns, found {len(hourcols)}")
    values = data[hourcols].apply(pd.to_numeric,errors="coerce")
    rawvalues = values.to_numpy()
    finite = rawvalues[np.isfinite(rawvalues)]
    sentinelrows.append({
        "year":year,
        "minus_codes":int((finite<=-900).sum()),
        "above_1000":int((finite>1000).sum()),
        "small_negative":int(((finite<0)&(finite>-900)).sum())
    })
    values = values.mask(values<=-900)
    frame = pd.DataFrame({
        "site":pd.to_numeric(data[sitecol],errors="coerce").astype("Int64").astype("string").str.zfill(6),
        "date":pd.to_datetime(data[datecol],errors="coerce"),
        "province":data[provincecol].astype("string").str.strip().str.upper(),
        "lat":pd.to_numeric(data[latcol],errors="coerce"),
        "lon":pd.to_numeric(data[loncol],errors="coerce"),
        "method":data[methodcol].astype("string").str.strip(),
        "valid_hours":values.notna().sum(axis=1),
        "pm":values.mean(axis=1)
    })
    frame = frame[frame["date"].dt.month.between(6,10)&frame["site"].notna()].copy()
    frame["year"] = year
    dailyrows.append(frame)
canadamethod = pd.concat(dailyrows,ignore_index=True)
usable = canadamethod[canadamethod["valid_hours"]>=18].copy()
methodcheck = usable.groupby(["site","date"],as_index=False).agg(
    rows=("method","size"),
    methods=("method","nunique"),
    minimum=("pm","min"),
    maximum=("pm","max")
)
methodcheck["range"] = methodcheck["maximum"]-methodcheck["minimum"]
canadadaily = usable.groupby(["site","date"],as_index=False).agg(
    pm=("pm","mean"),
    province=("province","first"),
    lat=("lat","median"),
    lon=("lon","median"),
    methods=("method","nunique"),
    valid_hours=("valid_hours","max")
)
canadadaily["year"] = canadadaily["date"].dt.year
naps_value_audit = pd.DataFrame(sentinelrows)
multi = methodcheck[methodcheck["rows"]>1]
print("Station-days:",len(canadadaily))
print("Raw stations:",canadadaily["site"].nunique())
print("Multi-method days:",len(multi))
display(naps_value_audit)
display(canadadaily.groupby("province",as_index=False).agg(
    raw_sites=("site","nunique"),
    site_days=("date","size")
).sort_values("raw_sites",ascending=False))

Station-days: 228371
Raw stations: 258
Multi-method days: 350


,year,minus_codes,above_1000,small_negative
0,2018,296822,12,441
1,2019,433799,13,328
2,2020,281143,0,0
3,2021,101085,4,0
4,2022,408175,2,0
5,2023,724810,16,0
6,2024,273278,0,5


,province,raw_sites,site_days
1,BC,65,57003
0,AB,52,44030
10,QC,50,43792
8,ON,47,42903
3,NB,10,10269
5,NS,7,7104
4,NL,6,6129
11,SK,6,5377
2,MB,5,3994
6,NT,5,3858


In [14]:
# Qualify Canadian candidates
canada_station_years = canadadaily.groupby(
    ["site","province","year"],as_index=False
).agg(days=("date","nunique"))

canada_feasibility_stats = canada_station_years.groupby(
    ["site","province"],as_index=False
).agg(
    years=("year","nunique"),
    days=("days","median"),
    total=("days","sum")
)
canada_feasibility_stats["complete"] = canada_feasibility_stats["total"]/(7*153)
canada_feasibility_stats["qualified"] = (
    canada_feasibility_stats["years"].ge(5)
    &canada_feasibility_stats["days"].ge(MIN_SEASON_DAYS)
)
feasibilityprovinceaudit = canada_feasibility_stats.groupby(
    "province",as_index=False
).agg(
    raw_sites=("site","size"),
    qualified_sites=("qualified","sum"),
    median_seasons=("years","median"),
    median_days=("days","median"),
    median_completeness=("complete","median")
)
feasibilityprovinceaudit["qualification_rate"] = (
    feasibilityprovinceaudit["qualified_sites"]/feasibilityprovinceaudit["raw_sites"]
)

qualificationcoverage = canada_station_years[
    canada_station_years["year"].between(QUALIFICATION_START_YEAR,QUALIFICATION_END_YEAR)
].copy()
canada_candidate_stats = qualificationcoverage.groupby(
    ["site","province"],as_index=False
).agg(
    years=("year","nunique"),
    days=("days","median"),
    total=("days","sum")
)
qualificationseasons = QUALIFICATION_END_YEAR-QUALIFICATION_START_YEAR+1
canada_candidate_stats["complete"] = canada_candidate_stats["total"]/(qualificationseasons*153)
canada_candidate_stats["qualified"] = (
    canada_candidate_stats["years"].ge(MIN_QUALIFICATION_YEARS)
    &canada_candidate_stats["days"].ge(MIN_SEASON_DAYS)
)
candidateprovinceaudit = canada_candidate_stats.groupby(
    "province",as_index=False
).agg(
    raw_sites=("site","size"),
    qualified_sites=("qualified","sum"),
    median_seasons=("years","median"),
    median_days=("days","median"),
    median_completeness=("complete","median")
)
candidateprovinceaudit["qualification_rate"] = (
    candidateprovinceaudit["qualified_sites"]/candidateprovinceaudit["raw_sites"]
)

qualifiedids = canada_candidate_stats.loc[
    canada_candidate_stats["qualified"],["site","province"]
]
qualifiedyears = canada_station_years.merge(
    qualifiedids,
    on=["site","province"]
).groupby(["province","year"],as_index=False).agg(
    qualified_sites=("site","nunique"),
    median_days=("days","median")
)

checks = {
    "Feasibility audit spans 2018-2024":canada_station_years["year"].min()==2018 and canada_station_years["year"].max()==2014+10,
    "Prospective qualification starts in 2018":qualificationcoverage["year"].min()==QUALIFICATION_START_YEAR,
    "Prospective qualification ends before validation":qualificationcoverage["year"].max()==QUALIFICATION_END_YEAR,
    "No 2023-2024 candidate qualification data":not qualificationcoverage["year"].isin([VALIDATION_YEAR,TEST_YEAR]).any(),
    "British Columbia budget feasible":int(candidateprovinceaudit.loc[candidateprovinceaudit["province"].eq("BC"),"qualified_sites"].iloc[0])>=SENSOR_BUDGET,
    "Ontario budget feasible":int(candidateprovinceaudit.loc[candidateprovinceaudit["province"].eq("ON"),"qualified_sites"].iloc[0])>=SENSOR_BUDGET
}
canadacoveragechecks = pd.DataFrame({"check":checks.keys(),"passed":checks.values()})

display(feasibilityprovinceaudit.sort_values("qualified_sites",ascending=False).round(4))
display(candidateprovinceaudit[candidateprovinceaudit["province"].isin(["BC","ON","AB"])].round(4))
display(qualifiedyears[qualifiedyears["province"].isin(["BC","ON","AB"])].round(2))
display(canadacoveragechecks)

if not canadacoveragechecks["passed"].all():
    failed = canadacoveragechecks.loc[~canadacoveragechecks["passed"],"check"].tolist()
    raise ValueError("Failed NAPS qualification checks: "+", ".join(failed))

,province,raw_sites,qualified_sites,median_seasons,median_days,median_completeness,qualification_rate
1,BC,65,55,7.0,150.00,0.9505,0.8462
0,AB,52,42,6.0,151.25,0.8473,0.8077
8,ON,47,42,7.0,150.00,0.9711,0.8936
10,QC,50,40,7.0,149.00,0.9510,0.8000
3,NB,10,10,7.0,153.00,0.9907,1.0000
5,NS,7,7,7.0,150.00,0.9542,1.0000
4,NL,6,6,7.0,151.00,0.9692,1.0000
11,SK,6,6,6.0,151.75,0.8361,1.0000
2,MB,5,4,7.0,137.00,0.8422,0.8000
6,NT,5,4,7.0,142.00,0.8599,0.8000


,province,raw_sites,qualified_sites,median_seasons,median_days,median_completeness,qualification_rate
0,AB,51,44,5.0,152.0,0.9778,0.8627
1,BC,62,55,5.0,150.5,0.9634,0.8871
8,ON,46,43,5.0,151.0,0.9765,0.9348


,province,year,qualified_sites,median_days
0,AB,2018,44,150.0
1,AB,2019,44,152.0
2,AB,2020,44,151.0
3,AB,2021,44,152.0
4,AB,2022,42,152.0
5,AB,2023,40,151.5
6,AB,2024,18,151.0
7,BC,2018,52,151.0
8,BC,2019,55,149.0
9,BC,2020,55,150.0


,check,passed
0,Feasibility audit spans 2018-2024,True
1,Prospective qualification starts in 2018,True
2,Prospective qualification ends before validation,True
3,No 2023-2024 candidate qualification data,True
4,British Columbia budget feasible,True
5,Ontario budget feasible,True


In [15]:
# Freeze transfer feasibility
napsrows = []
for code,name,decision in [
    ("BC","British Columbia","selected main transfer"),
    ("ON","Ontario","selected second transfer"),
    ("AB","Alberta","robustness candidate")
]:
    row = feasibilityprovinceaudit[feasibilityprovinceaudit["province"].eq(code)].iloc[0]
    napsrows.append({
        "region":name,
        "country":"Canada",
        "raw_sites":int(row.raw_sites),
        "qualified_sites":int(row.qualified_sites),
        "median_days":row.median_days,
        "median_completeness":row.median_completeness,
        "qualification_rate":row.qualification_rate,
        "source":"NAPS",
        "decision":decision,
        "reproduced_here":True
    })

prior_europe_audit = pd.DataFrame([
    {
        "region":"Portugal",
        "country":"Portugal",
        "raw_sites":32,
        "qualified_sites":21,
        "median_days":143.75,
        "median_completeness":.7428,
        "qualification_rate":.6562,
        "source":"EEA",
        "decision":"feasible secondary audit",
        "reproduced_here":False,
        "provenance_note":"Prior audit retained for continuity; exact access date was not recorded."
    },
    {
        "region":"Spain",
        "country":"Spain",
        "raw_sites":np.nan,
        "qualified_sites":np.nan,
        "median_days":np.nan,
        "median_completeness":np.nan,
        "qualification_rate":np.nan,
        "source":"EEA",
        "decision":"retrieval unresolved",
        "reproduced_here":False,
        "provenance_note":"Prior live retrieval unresolved; no zero-site claim is made."
    },
    {
        "region":"Greece",
        "country":"Greece",
        "raw_sites":np.nan,
        "qualified_sites":np.nan,
        "median_days":np.nan,
        "median_completeness":np.nan,
        "qualification_rate":np.nan,
        "source":"EEA",
        "decision":"retrieval unresolved",
        "reproduced_here":False,
        "provenance_note":"Prior live retrieval unresolved; no zero-site claim is made."
    }
])

transfer_feasibility = pd.concat([
    pd.DataFrame(napsrows),
    pd.DataFrame([us_audits["Washington"]["summary"],us_audits["Oregon"]["summary"]]),
    pd.DataFrame([england_summary]),
    prior_europe_audit.drop(columns="provenance_note")
],ignore_index=True)

display(transfer_feasibility.round(4))

,region,country,raw_sites,qualified_sites,median_days,median_completeness,qualification_rate,source,decision,reproduced_here
0,British Columbia,Canada,65.0,55.0,150.00,0.9505,0.8462,NAPS,selected main transfer,True
1,Ontario,Canada,47.0,42.0,150.00,0.9711,0.8936,NAPS,selected second transfer,True
2,Alberta,Canada,52.0,42.0,151.25,0.8473,0.8077,NAPS,robustness candidate,True
3,Washington,United States,26.0,17.0,148.50,0.9108,0.6538,EPA AirData,screened out,True
4,Oregon,United States,18.0,12.0,104.50,0.4958,0.6667,EPA AirData,screened out,True
5,England,United Kingdom,87.0,53.0,151.00,0.8413,0.6092,UK-AIR AURN,feasible external audit,True
6,Portugal,Portugal,32.0,21.0,143.75,0.7428,0.6562,EEA,feasible secondary audit,False
7,Spain,Spain,NaN,NaN,NaN,NaN,NaN,EEA,retrieval unresolved,False
8,Greece,Greece,NaN,NaN,NaN,NaN,NaN,EEA,retrieval unresolved,False


In [16]:
# Define Canadian region tools
def build_region(data,province):
    block = data[["province","site","date","pm","lat","lon"]].copy()
    block["province"] = block["province"].astype(str).str.upper().str.strip()
    block["site"] = block["site"].astype("string").str.zfill(6)
    block["date"] = pd.to_datetime(block["date"],errors="coerce")
    for column in ["pm","lat","lon"]:
        block[column] = pd.to_numeric(block[column],errors="coerce")
    keep = (
        block["province"].eq(province)
        &block["date"].notna()
        &block["pm"].notna()
        &block["date"].dt.year.between(QUALIFICATION_START_YEAR,TEST_YEAR)
        &block["date"].dt.month.between(6,10)
    )
    block = block[keep].copy()
    ids = canada_candidate_stats.loc[
        canada_candidate_stats["province"].eq(province)&canada_candidate_stats["qualified"],
        "site"
    ].astype(str).tolist()
    if len(ids)<SENSOR_BUDGET:
        raise ValueError(f"{province}: fewer than {SENSOR_BUDGET} training-qualified candidates")
    block = block[block["site"].isin(ids)].copy()
    duplicates = int(block.duplicated(["site","date"]).sum())
    block = block.groupby(["site","date"],as_index=False).agg(
        pm=("pm","mean"),
        lat=("lat","median"),
        lon=("lon","median")
    )
    trainingblock = block[block["date"].dt.year<=QUALIFICATION_END_YEAR].copy()
    nodes = trainingblock.groupby("site",as_index=False).agg(
        lat=("lat","median"),
        lon=("lon","median")
    ).set_index("site").reindex(ids).reset_index()
    matrix = block.pivot(index="date",columns="site",values="pm").reindex(columns=ids).sort_index()
    matrix.index = pd.DatetimeIndex(matrix.index)
    years = np.asarray(matrix.index.year)
    train = matrix.loc[(years>=QUALIFICATION_START_YEAR)&(years<=QUALIFICATION_END_YEAR)].copy()
    valid = matrix.loc[years==VALIDATION_YEAR].copy()
    test = matrix.loc[years==TEST_YEAR].copy()
    audit = {
        "province":province,
        "qualified_sites":len(ids),
        "qualification_start_year":QUALIFICATION_START_YEAR,
        "qualification_end_year":QUALIFICATION_END_YEAR,
        "site_days":len(block),
        "duplicate_input_days":duplicates,
        "matrix_rows":len(matrix),
        "train_rows":len(train),
        "validation_rows":len(valid),
        "test_rows":len(test),
        "missing_coordinates":int(nodes[["lat","lon"]].isna().any(axis=1).sum()),
        "train_missing_rate":float(train.isna().mean().mean())
    }
    return {
        "province":province,
        "daily":block,
        "stats":canada_candidate_stats[canada_candidate_stats["province"].eq(province)].copy(),
        "ids":ids,
        "nodes":nodes,
        "matrix":matrix,
        "train":train,
        "valid":valid,
        "test":test,
        "audit":audit
    }

In [17]:
# Build Canadian province matrices
canada_regions = {province:build_region(canadadaily,province) for province in ["BC","ON"]}
canada_region_audit = pd.DataFrame([canada_regions[province]["audit"] for province in ["BC","ON"]])
checks = {
    "British Columbia budget feasible":len(canada_regions["BC"]["ids"])>=SENSOR_BUDGET,
    "Ontario budget feasible":len(canada_regions["ON"]["ids"])>=SENSOR_BUDGET,
    "Candidate IDs come from training qualification":all(
        set(canada_regions[province]["ids"])==set(
            canada_candidate_stats.loc[
                canada_candidate_stats["province"].eq(province)&canada_candidate_stats["qualified"],
                "site"
            ].astype(str)
        )
        for province in ["BC","ON"]
    ),
    "Coordinates complete":canada_region_audit["missing_coordinates"].eq(0).all(),
    "Validation excluded from graph training":all(
        canada_regions[province]["train"].index.year.max()==QUALIFICATION_END_YEAR
        for province in ["BC","ON"]
    )
}
canadaregionchecks = pd.DataFrame({"check":checks.keys(),"passed":checks.values()})
display(canada_region_audit.round(4))
display(canadaregionchecks)
if not canadaregionchecks["passed"].all():
    failed = canadaregionchecks.loc[~canadaregionchecks["passed"],"check"].tolist()
    raise ValueError("Failed Canadian region checks: "+", ".join(failed))

,province,qualified_sites,qualification_start_year,qualification_end_year,site_days,duplicate_input_days,matrix_rows,train_rows,validation_rows,test_rows,missing_coordinates,train_missing_rate
0,BC,55,2018,2022,54684,0,1071,765,153,153,0,0.0710
1,ON,43,2018,2022,42547,0,1071,765,153,153,0,0.0417


,check,passed
0,British Columbia budget feasible,True
1,Ontario budget feasible,True
2,Candidate IDs come from training qualification,True
3,Coordinates complete,True
4,Validation excluded from graph training,True


In [18]:
# Define transfer graph tools
def graph_candidate(region,k,q,sigma_mult,minoverlap=60,return_graph=False):
    ids = list(region["ids"])
    nodes = region["nodes"].copy()
    nodes["site"] = nodes["site"].astype(str)
    nodes = nodes.drop_duplicates("site").set_index("site").reindex(ids).reset_index()
    train = region["train"].reindex(columns=ids)
    valid = region["valid"].reindex(columns=ids)
    n = len(ids)
    k = min(int(k),n-1)
    coordinates = np.radians(nodes[["lat","lon"]].to_numpy(dtype=float))
    distance = 6371.0088*haversine_distances(coordinates)
    neighbors = np.argsort(distance,axis=1)[:,1:k+1]
    mask = np.zeros((n,n),dtype=bool)
    mask[np.repeat(np.arange(n),k),neighbors.reshape(-1)] = True
    mask = mask|mask.T
    np.fill_diagonal(mask,False)
    presence = train.notna().astype(np.int32)
    overlap = (presence.T@presence).to_numpy()
    correlation = train.corr(min_periods=minoverlap).fillna(0).to_numpy()
    correlation = np.clip(correlation,-1,1)
    np.fill_diagonal(correlation,1)
    upper = np.triu(mask,1)
    sigma0 = float(np.median(distance[upper]))
    sigma = sigma0*float(sigma_mult)
    W = np.where(mask,np.exp(-(distance/sigma)**2)*np.maximum(correlation,0)**float(q),0)
    W = (W+W.T)/2
    np.fill_diagonal(W,0)
    rawparts,_ = connected_components(csr_matrix(W>0),directed=False)
    prune = GRAPH_PRUNE_RATIO*float(W.max())
    parts,_ = connected_components(csr_matrix(W>prune),directed=False)
    degree = W.sum(axis=1)
    L = np.diag(degree)-W
    values = np.maximum(np.linalg.eigvalsh(L),0)
    tol = np.finfo(float).eps*n*max(float(values[-1]),1)
    positivedegree = degree[degree>0]
    mediandegree = float(np.median(positivedegree)) if len(positivedegree) else np.nan
    lambda1relative = float(values[1]/values[-1]) if values[-1]>0 else 0
    minimumdegreerelative = float(degree.min()/mediandegree) if mediandegree>0 else 0
    X = valid.to_numpy(dtype=float)
    observed = np.isfinite(X)
    top = np.nan_to_num(X,nan=0)@W.T
    bottom = observed.astype(float)@W.T
    prediction = np.divide(top,bottom,out=np.full_like(top,np.nan),where=bottom>0)
    use = observed&np.isfinite(prediction)
    error = prediction[use]-X[use]
    row = {
        "k":k,
        "q":float(q),
        "sigma_mult":float(sigma_mult),
        "sigma0_km":sigma0,
        "sigma_km":sigma,
        "geographic_edges":int(upper.sum()),
        "weighted_edges":int(np.count_nonzero(np.triu(W>0,1))),
        "raw_components":int(rawparts),
        "robust_components":int(parts),
        "robust_edge_ratio":GRAPH_PRUNE_RATIO,
        "numerical_zeros":int((values<=tol).sum()),
        "lambda1":float(values[1]),
        "lambda1_relative":lambda1relative,
        "minimum_degree":float(degree.min()),
        "minimum_degree_relative":minimumdegreerelative,
        "validation_mae":float(np.mean(np.abs(error))),
        "validation_rmse":float(np.sqrt(np.mean(error**2))),
        "validation_coverage":float(use.sum()/observed.sum())
    }
    if not return_graph:
        return row,None
    values,vectors = np.linalg.eigh(L)
    values = np.maximum(values,0)
    graph = {
        "ids":ids,
        "nodes":nodes,
        "distance":distance,
        "mask":mask,
        "overlap":overlap,
        "correlation":correlation,
        "W":W,
        "degree":degree,
        "L":L,
        "values":values,
        "vectors":vectors,
        "prune":prune,
        "prune_ratio":GRAPH_PRUNE_RATIO
    }
    return row,graph

In [19]:
# Tune Canadian transfer graphs
canada_graph_tuning = {}
canada_selected_graphs = {}
auditrows = []
for province in ["BC","ON"]:
    rows = []
    for k,q,scale in product([8,10,12,15,20,25,30],[.5,1,2],[1,1.5,2,3,4,6,8]):
        if k<len(canada_regions[province]["ids"]):
            rows.append(graph_candidate(canada_regions[province],k,q,scale)[0])
    tuning = pd.DataFrame(rows)
    viable = tuning[
        tuning["raw_components"].eq(1)
        &tuning["robust_components"].eq(1)
        &tuning["numerical_zeros"].eq(1)
        &tuning["validation_coverage"].ge(.95)
    ].copy()
    canada_graph_tuning[province] = tuning
    if viable.empty:
        raise RuntimeError(
            f"No graph for {province} remains connected after pruning edges below "
            f"{GRAPH_PRUNE_RATIO:g} of the maximum weight"
        )
    best = viable["validation_mae"].min()
    near = viable[viable["validation_mae"]<=1.01*best]
    choice = near.sort_values(
        ["k","sigma_mult","validation_rmse","q"],
        ascending=[True,True,True,False]
    ).iloc[0]
    selected,graph = graph_candidate(
        canada_regions[province],
        int(choice.k),
        float(choice.q),
        float(choice.sigma_mult),
        return_graph=True
    )
    canada_selected_graphs[province] = graph
    baseline = tuning[
        tuning["k"].eq(10)
        &tuning["q"].eq(2)
        &tuning["sigma_mult"].eq(1)
    ].iloc[0]
    for label,row in [("California fixed",baseline),("Selected",pd.Series(selected))]:
        auditrows.append({
            "province":province,
            "graph":label,
            "k":row.k,
            "q":row.q,
            "sigma_mult":row.sigma_mult,
            "sigma_km":row.sigma_km,
            "components":row.robust_components,
            "robust_edge_ratio":row.robust_edge_ratio,
            "lambda1":row.lambda1,
            "lambda1_relative":row.lambda1_relative,
            "minimum_degree":row.minimum_degree,
            "minimum_degree_relative":row.minimum_degree_relative,
            "validation_mae":row.validation_mae
        })
canada_graph_audit = pd.DataFrame(auditrows)
selectedchecks = canada_graph_audit[canada_graph_audit["graph"].eq("Selected")]
if set(canada_selected_graphs)!={"BC","ON"}:
    raise ValueError("Missing selected Canadian graph")
if not selectedchecks["components"].eq(1).all():
    raise ValueError("Selected graph fails robust connectivity check")
display(canada_graph_audit.round(8))

,province,graph,k,q,sigma_mult,sigma_km,components,robust_edge_ratio,lambda1,lambda1_relative,minimum_degree,minimum_degree_relative,validation_mae
0,BC,California fixed,10.0,2.0,1.0,61.108264,2.0,0.000001,0.000000,0.000000,0.000000,0.000000,3.431641
1,BC,Selected,8.0,0.5,2.0,104.156367,1.0,0.000001,0.000540,0.000034,0.000549,0.000099,3.332920
2,ON,California fixed,10.0,2.0,1.0,80.592430,2.0,0.000001,0.000000,0.000000,0.000000,0.000000,1.993845
3,ON,Selected,8.0,2.0,2.0,151.516532,1.0,0.000001,0.000166,0.000017,0.000163,0.000040,2.088577


In [20]:
# Audit final Canadian spectra
canada_spectral_final = {}
rows = []
for province in ["BC","ON"]:
    region = canada_regions[province]
    graph = canada_selected_graphs[province]
    X = region["train"].to_numpy(dtype=float)
    means = np.nanmean(X,axis=0)
    filled = np.where(np.isnan(X),means[None,:],X)
    coefficients = filled@graph["vectors"]
    total = np.sum(coefficients**2,axis=0)
    centered = filled-filled.mean(axis=0,keepdims=True)
    centeredcoefficients = centered@graph["vectors"]
    spatial = np.sum(centeredcoefficients**2,axis=0)
    total = np.cumsum(total)/total.sum()
    spatial = np.cumsum(spatial)/spatial.sum()
    K95 = int(np.searchsorted(total,.95)+1)
    centeredK95 = int(np.searchsorted(spatial,.95)+1)
    canada_spectral_final[province] = {
        "K95":K95,
        "centered_K95":centeredK95,
        "total_fraction":total,
        "centered_fraction":spatial
    }
    for K in [value for value in [10,15,20,25,30,35,40] if value<len(graph["ids"])]:
        rows.append({
            "province":province,
            "K":K,
            "total_energy":total[K-1],
            "centered_energy":spatial[K-1],
            "lambda_K":graph["values"][K-1]
        })
canada_spectral_audit = pd.DataFrame(rows)
canada_spectral_summary = pd.DataFrame([
    {
        "province":province,
        "K95_total":values["K95"],
        "K95_centered":values["centered_K95"]
    }
    for province,values in canada_spectral_final.items()
])
display(canada_spectral_summary)
display(canada_spectral_audit.round(6))

,province,K95_total,K95_centered
0,BC,19,23
1,ON,15,34


,province,K,total_energy,centered_energy,lambda_K
0,BC,10,0.830681,0.790270,0.520248
1,BC,15,0.916364,0.896529,1.578253
2,BC,20,0.958110,0.949068,2.575847
3,BC,25,0.964900,0.957388,4.607864
4,BC,30,0.971960,0.965907,6.109127
5,BC,35,0.977484,0.972494,7.852584
6,BC,40,0.984472,0.981068,9.250077
7,ON,10,0.937691,0.787373,1.359244
8,ON,15,0.950901,0.821296,2.574237
9,ON,20,0.960008,0.856834,3.388498


In [21]:
# Load Statistics Canada target data
target_specs = {
    "BC_Vancouver":{"province":"BC","cma":"933","expected_population":2642825},
    "BC_Kelowna":{"province":"BC","cma":"915","expected_population":222162},
    "ON_Toronto":{"province":"ON","cma":"535","expected_population":6202225}
}
statdir = Path("statcan")
statdir.mkdir(exist_ok=True)
ctzip = statdir/"lct_000b21a_e.zip"
tablezip = statdir/"98100014-eng.zip"
if not ctzip.exists():
    response = requests.get(
        "https://www12.statcan.gc.ca/census-recensement/2021/geo/sip-pis/boundary-limites/files-fichiers/lct_000b21a_e.zip",
        timeout=180
    )
    response.raise_for_status()
    if not zipfile.is_zipfile(io.BytesIO(response.content)):
        raise ValueError("Invalid Statistics Canada census-tract archive")
    ctzip.write_bytes(response.content)
ctdir = statdir/"ct"
ctdir.mkdir(exist_ok=True)
if not list(ctdir.glob("*.shp")):
    with zipfile.ZipFile(ctzip) as bundle:
        bundle.extractall(ctdir)
if not tablezip.exists():
    urls = []
    try:
        response = requests.get(
            "https://www150.statcan.gc.ca/t1/wds/rest/getFullTableDownloadCSV/98100014/en",
            timeout=60
        )
        if response.ok and response.json().get("status")=="SUCCESS":
            urls.append(response.json()["object"])
    except Exception:
        pass
    urls.append("https://www150.statcan.gc.ca/n1/tbl/csv/98100014-eng.zip")
    for url in urls:
        try:
            response = requests.get(url,timeout=180)
            if response.ok and zipfile.is_zipfile(io.BytesIO(response.content)):
                tablezip.write_bytes(response.content)
                break
        except Exception:
            pass
if not tablezip.exists():
    raise RuntimeError("Statistics Canada population table unavailable")
canada_ct = gpd.read_file(list(ctdir.glob("*.shp"))[0])
with zipfile.ZipFile(tablezip) as bundle:
    names = [
        name for name in bundle.namelist()
        if name.lower().endswith(".csv") and "metadata" not in name.lower()
    ]
    census_table = pd.read_csv(io.BytesIO(bundle.read(names[0])),low_memory=False)

In [22]:
# Audit Canadian census targets
norm = lambda value:re.sub(r"[^a-z0-9]","",str(value).lower())
ccols = {norm(column):column for column in census_table.columns}
gcols = {norm(column):column for column in canada_ct.columns}
dguidcol = ccols.get("dguid")
popcols = [
    column for column in census_table.columns
    if "population2021" in norm(column) and "symbol" not in norm(column)
]
ctuidcol = gcols.get("ctuid")
geodguid = gcols.get("dguid")
if dguidcol is None or len(popcols)!=1 or ctuidcol is None:
    raise RuntimeError("Unexpected Statistics Canada schema")
official = census_table[[dguidcol,popcols[0]]].copy()
official.columns = ["dguid","population"]
official["dguid"] = official["dguid"].astype(str).str.strip()
official["population"] = pd.to_numeric(
    official["population"].astype(str).str.replace(",","",regex=False),
    errors="coerce"
)
official["official_row"] = True
tracts = canada_ct.copy()
tracts["ctuid"] = tracts[ctuidcol].astype(str).str.strip()
tracts["dguid"] = tracts[geodguid].astype(str).str.strip() if geodguid else "2021S0507"+tracts["ctuid"]
tracts["cma"] = tracts["ctuid"].str[:3]
tracts = tracts[
    tracts["cma"].isin({spec["cma"] for spec in target_specs.values()})
].merge(official,on="dguid",how="left",validate="many_to_one")
rows = []
for key,spec in target_specs.items():
    block = tracts[tracts["cma"].eq(spec["cma"])]
    total = block["population"].sum(min_count=1)
    rows.append({
        "target":key,
        "tracts":len(block),
        "official_rows":int(block["official_row"].fillna(False).sum()),
        "missing_join_rows":int(block["official_row"].isna().sum()),
        "official_null_population":int((block["official_row"].eq(True)&block["population"].isna()).sum()),
        "published_population_sum":total,
        "expected_population":spec["expected_population"],
        "relative_error":abs(total-spec["expected_population"])/spec["expected_population"]
    })
canada_census_audit = pd.DataFrame(rows)
checks = {
    "Census joins complete":canada_census_audit["missing_join_rows"].eq(0).all(),
    "Published totals reproduced":canada_census_audit["relative_error"].max()<1e-10
}
canadacensuschecks = pd.DataFrame({"check":checks.keys(),"passed":checks.values()})
display(canada_census_audit)
display(canadacensuschecks)
if not canadacensuschecks["passed"].all():
    failed = canadacensuschecks.loc[~canadacensuschecks["passed"],"check"].tolist()
    raise ValueError("Failed Canadian census checks: "+", ".join(failed))

,target,tracts,official_rows,missing_join_rows,official_null_population,published_population_sum,expected_population,relative_error
0,BC_Vancouver,535,535,0,1,2642825.0,2642825,0.0
1,BC_Kelowna,52,52,0,0,222162.0,222162,0.0
2,ON_Toronto,1227,1227,0,0,6202225.0,6202225,0.0


,check,passed
0,Census joins complete,True
1,Published totals reproduced,True


In [23]:
# Build Canadian population targets
projected = tracts.to_crs(3347)
centers = gpd.GeoSeries(projected.geometry.centroid,index=projected.index,crs=3347).to_crs(4326)
tracts["center_lat"] = centers.y
tracts["center_lon"] = centers.x
canada_target_weights = {}
targetrows = []
weightrows = []
for key,spec in target_specs.items():
    block = tracts[
        tracts["cma"].eq(spec["cma"])
        &tracts["population"].notna()
    ].copy()
    graph = canada_selected_graphs[spec["province"]]
    ids = list(graph["ids"])
    nodes = graph["nodes"].set_index("site").reindex(ids).reset_index()
    distance = 6371.0088*haversine_distances(
        np.radians(block[["center_lat","center_lon"]].to_numpy(dtype=float)),
        np.radians(nodes[["lat","lon"]].to_numpy(dtype=float))
    )
    nearest = np.argmin(distance,axis=1)
    block["nearest_site"] = np.asarray(ids)[nearest]
    block["nearest_km"] = distance[np.arange(len(block)),nearest]
    weights = block.groupby("nearest_site")["population"].sum().reindex(ids).fillna(0)
    weights = weights/weights.sum()
    canada_target_weights[key] = weights
    support = int((weights>0).sum())
    effective = float(1/np.sum(weights.to_numpy()**2))
    eligible = support>=3 and effective>=2
    targetrows.append({
        "target":key,
        "province":spec["province"],
        "population_used":block["population"].sum(),
        "population_tracts":len(block),
        "positive_weight_stations":support,
        "effective_weight_support":effective,
        "median_nearest_km":block["nearest_km"].median(),
        "p95_nearest_km":block["nearest_km"].quantile(.95),
        "maximum_nearest_km":block["nearest_km"].max(),
        "eligible":eligible
    })
    weightrows.extend(
        {"target":key,"site":site,"weight":value}
        for site,value in weights.items() if value>0
    )
canada_target_audit = pd.DataFrame(targetrows)
canada_target_weight_table = pd.DataFrame(weightrows)
active_canada_targets = canada_target_audit.loc[canada_target_audit["eligible"],"target"].tolist()
checks = {
    "Vancouver eligible":"BC_Vancouver" in active_canada_targets,
    "Toronto eligible":"ON_Toronto" in active_canada_targets,
    "Kelowna excluded":"BC_Kelowna" not in active_canada_targets
}
canadatargetchecks = pd.DataFrame({"check":checks.keys(),"passed":checks.values()})
display(canada_target_audit.round(4))
display(canadatargetchecks)
print("Active targets:",active_canada_targets)
if not canadatargetchecks["passed"].all():
    failed = canadatargetchecks.loc[~canadatargetchecks["passed"],"check"].tolist()
    raise ValueError("Failed Canadian target checks: "+", ".join(failed))

,target,province,population_used,population_tracts,positive_weight_stations,effective_weight_support,median_nearest_km,p95_nearest_km,maximum_nearest_km,eligible
0,BC_Vancouver,BC,2642825.0,534,17,9.8905,3.9825,11.1822,19.8783,True
1,BC_Kelowna,BC,222162.0,52,2,1.0265,7.4367,20.7992,22.7956,False
2,ON_Toronto,ON,6202225.0,1227,14,8.4773,5.8828,17.0701,47.1638,True


,check,passed
0,Vancouver eligible,True
1,Toronto eligible,True
2,Kelowna excluded,True


Active targets: ['BC_Vancouver', 'ON_Toronto']


In [24]:
# Define target models
def build_target_model(key):
    spec = target_specs[key]
    graph = canada_selected_graphs[spec["province"]]
    ids = list(graph["ids"])
    K = int(canada_spectral_final[spec["province"]]["centered_K95"])
    weights = canada_target_weights[key].reindex(ids).fillna(0).to_numpy(dtype=float)
    scaled = graph["values"][:K]/graph["values"][K-1]
    basis = graph["vectors"][:,:K]
    decay = np.exp(-np.outer(np.array([0,.25,.5]),scaled))
    blocks = basis[:,None,:]*decay[None,:,:]
    g = np.exp(-scaled)*(basis.T@weights)
    info = np.einsum("vtk,vtl->vkl",blocks,blocks)
    beta = .01*np.trace(info.sum(axis=0))/K
    return {
        "target":key,
        "province":spec["province"],
        "ids":ids,
        "K":K,
        "beta":beta,
        "blocks":blocks,
        "info":info,
        "g":g,
        "weights":weights,
        "graph":graph
    }

def design_metrics(model,selected):
    B = model["blocks"][selected].reshape(-1,model["K"])
    g = model["g"]
    beta = model["beta"]
    normalizer = np.linalg.norm(g)
    exact = np.linalg.lstsq(B.T,g,rcond=None)[0]
    span = np.linalg.norm(B.T@exact-g)/normalizer
    M = beta*np.eye(model["K"])+B.T@B
    h = np.linalg.solve(M,g)
    coefficients = B@h
    mismatch = np.linalg.norm(B.T@coefficients-g)/normalizer
    amplification = np.sqrt(beta)*np.linalg.norm(coefficients)/normalizer
    risk = np.sqrt(mismatch**2+amplification**2)
    rank = int(np.linalg.matrix_rank(B))
    singular = np.linalg.svd(B,compute_uv=False)
    sigma = float(singular[-1]) if rank==model["K"] else 0
    return {
        "span_error":span,
        "mismatch":mismatch,
        "amplification":amplification,
        "risk":risk,
        "rank":rank,
        "sigma_min":sigma,
        "full_state_risk":np.sqrt(beta/(sigma**2+beta))
    }

In [25]:
# Define sensor design tools
def taps_order(model,budget=10):
    M = model["beta"]*np.eye(model["K"])
    selected = []
    available = np.ones(len(model["ids"]),dtype=bool)
    normalizer = model["g"]@model["g"]
    for step in range(budget):
        best = -1
        bestscore = np.inf
        for station in np.flatnonzero(available):
            score = model["beta"]*(model["g"]@np.linalg.solve(M+model["info"][station],model["g"]))/normalizer
            if score<bestscore-1e-14:
                best = int(station)
                bestscore = score
        selected.append(best)
        available[best] = False
        M = M+model["info"][best]
    return selected

def raw_order(model,budget=10):
    M = model["beta"]*np.eye(model["K"])
    selected = []
    available = np.ones(len(model["ids"]),dtype=bool)
    for step in range(budget):
        h = np.linalg.solve(M,model["g"])
        response = np.einsum("vtk,k->vt",model["blocks"],h)
        score = np.sum(response**2,axis=1)
        score[~available] = -np.inf
        best = int(np.argmax(score))
        selected.append(best)
        available[best] = False
        M = M+model["info"][best]
    return selected

def transfer_placements(model,budget=10):
    graph = model["graph"]
    taps = taps_order(model,budget)
    target = np.argsort(-model["weights"],kind="stable")[:budget].tolist()
    degree = np.argsort(-graph["degree"],kind="stable")[:budget].tolist()
    geographic = [int(np.argmin(graph["distance"].max(axis=1)))]
    while len(geographic)<budget:
        nearest = graph["distance"][:,geographic].min(axis=1)
        nearest[geographic] = -np.inf
        geographic.append(int(np.argmax(nearest)))
    M = model["beta"]*np.eye(model["K"])
    doptimal = []
    available = np.ones(len(model["ids"]),dtype=bool)
    for step in range(budget):
        best = -1
        bestscore = -np.inf
        for station in np.flatnonzero(available):
            sign,score = np.linalg.slogdet(M+model["info"][station])
            if sign>0 and score>bestscore+1e-12:
                best = int(station)
                bestscore = score
        doptimal.append(best)
        available[best] = False
        M = M+model["info"][best]
    basis = graph["vectors"][:,:model["K"]]
    _,_,pivots = qr(basis.T,pivoting=True,mode="economic")
    return {
        "TAPS":taps,
        "Raw response":raw_order(model,budget),
        "Target weight":target,
        "Weighted degree":degree,
        "Geographic coverage":geographic,
        "D-optimal":doptimal,
        "QR pivoting":[int(value) for value in pivots[:budget]]
    }

In [26]:
# Build model-based placements
canada_models = {}
tapsrows = []
placementrows = []
selectionrows = []
exactnessrows = []

for key in active_canada_targets:
    model = build_target_model(key)
    placements = transfer_placements(model,SENSOR_BUDGET)
    taps = placements["TAPS"]
    model["taps"] = taps
    canada_models[key] = model

    energy = np.sort(model["g"]**2)[::-1]
    effective = int(np.searchsorted(np.cumsum(energy)/energy.sum(),.95)+1)

    fullorder = taps_order(model,len(model["ids"]))
    targetscan = []
    for budget in range(1,len(fullorder)+1):
        selected = fullorder[:budget]
        metrics = design_metrics(model,selected)
        row = {
            "target":key,
            "budget":budget,
            "selected_sites":"|".join(np.asarray(model["ids"])[selected]),
            **metrics
        }
        targetscan.append(row)
        exactnessrows.append(row)

    targetscan = pd.DataFrame(targetscan)
    exactrows = targetscan[targetscan["span_error"].lt(EXACT_TOLERANCE)]
    fullrankrows = targetscan[targetscan["rank"].ge(model["K"])]

    exactbudget = int(exactrows["budget"].iloc[0]) if len(exactrows) else np.nan
    exactrank = int(exactrows["rank"].iloc[0]) if len(exactrows) else np.nan
    fullrankbudget = int(fullrankrows["budget"].iloc[0]) if len(fullrankrows) else np.nan
    exactseparation = bool(len(exactrows) and exactrank<model["K"])

    budgetmetrics = design_metrics(model,taps)
    tapsrows.append({
        "target":key,
        "province":model["province"],
        "K":model["K"],
        "beta":model["beta"],
        "target_effective_dimension":effective,
        "sensor_budget":SENSOR_BUDGET,
        "selected_sites":"|".join(np.asarray(model["ids"])[taps]),
        "exact_at_fixed_budget":budgetmetrics["span_error"]<EXACT_TOLERANCE,
        "taps_first_exact_budget":exactbudget,
        "rank_at_taps_first_exact":exactrank,
        "taps_first_full_rank_budget":fullrankbudget,
        "target_state_separation_at_taps_first_exact":exactseparation,
        **budgetmetrics
    })

    for method,selected in placements.items():
        placementrows.append({
            "target":key,
            "method":method,
            **design_metrics(model,selected)
        })
        selectionrows.append({
            "target":key,
            "method":method,
            "overlap_with_taps":len(set(selected)&set(taps)),
            "sites":"|".join(np.asarray(model["ids"])[selected])
        })

canada_taps_results = pd.DataFrame(tapsrows)
canada_taps_exactness_scan = pd.DataFrame(exactnessrows)
canada_placement_results = pd.DataFrame(placementrows)
canada_selection_results = pd.DataFrame(selectionrows)

checks = {
    "Fixed budget respected":canada_selection_results["sites"].str.split("|").str.len().eq(SENSOR_BUDGET).all(),
    "TAPS risks finite":np.isfinite(canada_taps_results["risk"]).all(),
    "Exactness scans complete":all(
        len(canada_taps_exactness_scan[canada_taps_exactness_scan["target"].eq(key)])
        ==len(canada_models[key]["ids"])
        for key in active_canada_targets
    ),
    "Span error nonincreasing":all(
        np.all(np.diff(
            canada_taps_exactness_scan.loc[
                canada_taps_exactness_scan["target"].eq(key),"span_error"
            ].to_numpy()
        )<=1e-10)
        for key in active_canada_targets
    ),
    "Rank nondecreasing":all(
        np.all(np.diff(
            canada_taps_exactness_scan.loc[
                canada_taps_exactness_scan["target"].eq(key),"rank"
            ].to_numpy()
        )>=0)
        for key in active_canada_targets
    )
}
tapsscanchecks = pd.DataFrame({"check":checks.keys(),"passed":checks.values()})

display(canada_taps_results.round(6))
display(canada_placement_results.sort_values(["target","risk"]).round(6))
display(tapsscanchecks)

if not tapsscanchecks["passed"].all():
    failed = tapsscanchecks.loc[~tapsscanchecks["passed"],"check"].tolist()
    raise ValueError("Failed TAPS scan checks: "+", ".join(failed))

,target,province,K,beta,target_effective_dimension,sensor_budget,selected_sites,exact_at_fixed_budget,taps_first_exact_budget,rank_at_taps_first_exact,taps_first_full_rank_budget,target_state_separation_at_taps_first_exact,span_error,mismatch,amplification,risk,rank,sigma_min,full_state_risk
0,BC_Vancouver,BC,23,0.026238,9,10,100125|100111|102103|101202|105001|100103|1001...,True,4,12,43,True,0.000000,0.033222,0.154546,0.158076,19,0,1.0
1,ON_Toronto,ON,34,0.024749,13,10,060445|060440|061603|065101|061502|060438|0604...,False,11,33,12,True,0.000091,0.092702,0.139072,0.167137,30,0,1.0


,target,method,span_error,mismatch,amplification,risk,rank,sigma_min,full_state_risk
0,BC_Vancouver,TAPS,0.000000,0.033222,0.154546,0.158076,19,0.000000,1.0
1,BC_Vancouver,Raw response,0.000000,0.032044,0.171875,0.174837,20,0.000000,1.0
2,BC_Vancouver,Target weight,0.000000,0.148116,0.178896,0.232255,19,0.000000,1.0
3,BC_Vancouver,Weighted degree,0.000000,0.148931,0.178808,0.232707,19,0.000000,1.0
4,BC_Vancouver,Geographic coverage,0.000000,0.946797,0.176444,0.963098,23,0.000018,1.0
5,BC_Vancouver,D-optimal,0.000000,0.994253,0.068751,0.996627,23,0.000000,1.0
6,BC_Vancouver,QR pivoting,0.000000,0.999998,0.001386,0.999999,23,0.000000,1.0
7,ON_Toronto,TAPS,0.000091,0.092702,0.139072,0.167137,30,0.000000,1.0
8,ON_Toronto,Raw response,0.000076,0.117384,0.153529,0.193262,30,0.000000,1.0
9,ON_Toronto,Target weight,0.000152,0.136946,0.144131,0.198816,30,0.000000,1.0


,check,passed
0,Fixed budget respected,True
1,TAPS risks finite,True
2,Exactness scans complete,True
3,Span error nonincreasing,True
4,Rank nondecreasing,True


In [27]:
# Benchmark model-based random placements
randomrows = []
for key in active_canada_targets:
    model = canada_models[key]
    n = len(model["ids"])
    if comb(n,SENSOR_BUDGET)<RANDOM_SETS:
        raise ValueError(f"{key}: fewer than {RANDOM_SETS} distinct sensor sets at budget {SENSOR_BUDGET}")
    rng = np.random.default_rng(RANDOM_SEED)
    seen = set()
    while len(seen)<RANDOM_SETS:
        seen.add(tuple(sorted(rng.choice(len(model["ids"]),SENSOR_BUDGET,replace=False).tolist())))
    for draw,selected in enumerate(sorted(seen)):
        randomrows.append({
            "target":key,
            "draw":draw,
            "sites":"|".join(np.asarray(model["ids"])[list(selected)]),
            **design_metrics(model,list(selected))
        })
canada_random_model = pd.DataFrame(randomrows)
canada_model_random_sets = canada_random_model[["target","draw","sites"]].copy()
random_model_summary = canada_random_model.groupby("target",as_index=False).agg(
    random_mean_risk=("risk","mean"),
    random_p05_risk=("risk",lambda values:np.quantile(values,.05)),
    random_p95_risk=("risk",lambda values:np.quantile(values,.95))
)
tapsrisk = canada_placement_results[
    canada_placement_results["method"].eq("TAPS")
].set_index("target")["risk"]
random_model_summary["taps_better_than_random_fraction"] = [
    float((canada_random_model.loc[
        canada_random_model["target"].eq(key),"risk"
    ]>tapsrisk[key]).mean())
    for key in random_model_summary["target"]
]
checks = {
    "Random model sets complete":canada_random_model.groupby("target").size().eq(RANDOM_SETS).all(),
    "Random model sets unique":canada_model_random_sets.groupby("target")["sites"].nunique().eq(RANDOM_SETS).all()
}
randommodelchecks = pd.DataFrame({"check":checks.keys(),"passed":checks.values()})
display(random_model_summary.round(6))
display(randommodelchecks)
if not randommodelchecks["passed"].all():
    raise ValueError("Model-based random-set audit failed")

,target,random_mean_risk,random_p05_risk,random_p95_risk,taps_better_than_random_fraction
0,BC_Vancouver,0.308208,0.203697,0.508175,1.0
1,ON_Toronto,0.514406,0.301212,0.854533,1.0


,check,passed
0,Random model sets complete,True
1,Random model sets unique,True


In [28]:
# Build held-out forecast protocols
canada_eval_threshold = {key:TARGET_COVERAGE_THRESHOLD for key in active_canada_targets}
canada_eval_protocol = {
    key:f"full{int(round(100*TARGET_COVERAGE_THRESHOLD))}"
    for key in active_canada_targets
}

def observed_target(matrix,weights,threshold):
    weights = weights[weights>0].copy()
    block = matrix.reindex(columns=weights.index)
    coverage = block.notna().mul(weights,axis=1).sum(axis=1)
    value = block.mul(weights,axis=1).sum(axis=1,min_count=1)/coverage.where(coverage>0)
    return value.where(coverage>=threshold),coverage

canada_forecast_samples = {}
rows = []
for key in active_canada_targets:
    model = canada_models[key]
    matrix = canada_regions[model["province"]]["matrix"].reindex(columns=model["ids"]).copy()
    matrix.index = pd.DatetimeIndex(matrix.index)
    weights = canada_target_weights[key].reindex(model["ids"]).fillna(0)
    truth,targetcoverage = observed_target(matrix,weights,canada_eval_threshold[key])
    available = set(matrix.index)
    windows = []
    for date in matrix.index:
        dates = [date+pd.Timedelta(days=offset) for offset in OBSERVATION_OFFSETS]
        if all(value in available and value.year==date.year for value in dates):
            block = matrix.loc[dates]
            windows.append([
                date,
                *dates,
                int(date.year),
                float(block.notna().mean().mean()),
                int(block.notna().all(axis=0).sum())
            ])
    frame = pd.DataFrame(
        windows,
        columns=["target_date","first_date","middle_date","latest_date","year","input_fraction","complete_stations"]
    )
    frame["truth"] = truth.reindex(frame["target_date"]).to_numpy(dtype=float)
    frame["coverage"] = targetcoverage.reindex(frame["target_date"]).to_numpy(dtype=float)
    canada_forecast_samples[key] = frame
    for year in range(QUALIFICATION_START_YEAR,TEST_YEAR+1):
        subset = frame[(frame["year"]==year)&frame["truth"].notna()]
        rows.append({
            "target":key,
            "protocol":canada_eval_protocol[key],
            "year":year,
            "truth_days":len(subset),
            "median_coverage":subset["coverage"].median() if len(subset) else np.nan,
            "median_input_fraction":subset["input_fraction"].median() if len(subset) else np.nan,
            "min_complete_stations":int(subset["complete_stations"].min()) if len(subset) else 0
        })
canada_forecast_audit = pd.DataFrame(rows)
testcoverage = canada_forecast_audit[
    canada_forecast_audit["year"].isin([VALIDATION_YEAR,TEST_YEAR])
].pivot(index="target",columns="year",values="truth_days")
if not (testcoverage>0).all().all():
    raise ValueError("Uniform pre-specified target threshold leaves no validation or test dates")
display(canada_forecast_audit.round(4))
display(testcoverage)

,target,protocol,year,truth_days,median_coverage,median_input_fraction,min_complete_stations
0,BC_Vancouver,full85,2018,145,1.0000,0.9030,39
1,BC_Vancouver,full85,2019,141,1.0000,0.9091,41
2,BC_Vancouver,full85,2020,141,1.0000,0.9636,40
3,BC_Vancouver,full85,2021,148,1.0000,0.9636,47
4,BC_Vancouver,full85,2022,147,1.0000,0.9273,43
5,BC_Vancouver,full85,2023,147,1.0000,0.9394,42
6,BC_Vancouver,full85,2024,149,1.0000,0.9152,44
7,ON_Toronto,full85,2018,0,NaN,NaN,0
8,ON_Toronto,full85,2019,141,1.0000,0.9767,37
9,ON_Toronto,full85,2020,143,1.0000,0.9845,36


year,2023,2024
target,,
BC_Vancouver,147,149
ON_Toronto,138,126


In [29]:
# Define forecast tools
def prediction_metrics(prediction,truth):
    prediction = np.asarray(prediction,dtype=float)
    truth = np.asarray(truth,dtype=float)
    valid = np.isfinite(prediction)&np.isfinite(truth)
    prediction = prediction[valid]
    truth = truth[valid]
    if not len(truth):
        return {"n":0,"mae":np.nan,"rmse":np.nan,"bias":np.nan,"correlation":np.nan}
    error = prediction-truth
    correlation = np.corrcoef(prediction,truth)[0,1] if len(truth)>1 and np.std(prediction)>0 and np.std(truth)>0 else np.nan
    return {
        "n":len(truth),
        "mae":float(np.mean(np.abs(error))),
        "rmse":float(np.sqrt(np.mean(error**2))),
        "bias":float(np.mean(error)),
        "correlation":float(correlation) if np.isfinite(correlation) else np.nan
    }

def calibrate(prediction,truth):
    prediction = np.asarray(prediction,dtype=float)
    truth = np.asarray(truth,dtype=float)
    valid = np.isfinite(prediction)&np.isfinite(truth)
    X = np.column_stack([np.ones(valid.sum()),prediction[valid]])
    return np.linalg.lstsq(X,truth[valid],rcond=None)[0]

In [30]:
# Define forecast features
def forecast_features(key,frame,selected):
    model = canada_models[key]
    matrix = canada_regions[model["province"]]["matrix"].reindex(columns=model["ids"])
    sites = [model["ids"][index] for index in selected]
    rows = []
    for row in frame.itertuples():
        block = matrix.loc[
            [row.first_date,row.middle_date,row.latest_date],sites
        ].to_numpy(dtype=float).T.reshape(-1)
        angle = 2*np.pi*row.target_date.dayofyear/365.25
        rows.append(np.r_[block,np.sin(angle),np.cos(angle)])
    return np.asarray(rows,dtype=float)

def latest_average(key,frame,selected):
    model = canada_models[key]
    matrix = canada_regions[model["province"]]["matrix"].reindex(columns=model["ids"])
    sites = [model["ids"][index] for index in selected]
    return np.asarray([
        matrix.loc[row.latest_date,sites].mean()
        for row in frame.itertuples()
    ],dtype=float)

def graph_predictions(key,frame,selected):
    model = canada_models[key]
    matrix = canada_regions[model["province"]]["matrix"].reindex(columns=model["ids"])
    sites = [model["ids"][index] for index in selected]
    design = model["blocks"][selected].reshape(-1,model["K"])
    predictions = []
    counts = []
    for row in frame.itertuples():
        y = matrix.loc[
            [row.first_date,row.middle_date,row.latest_date],sites
        ].to_numpy(dtype=float).T.reshape(-1)
        observed = np.isfinite(y)
        A = design[observed]
        M = model["beta"]*np.eye(model["K"])+A.T@A
        state = np.linalg.solve(M,A.T@y[observed])
        predictions.append(model["g"]@state)
        counts.append(int(observed.sum()))
    return np.asarray(predictions),np.asarray(counts)

In [31]:
# Define bootstrap tools
def bootdiff(frame,a,b,reps=BOOTSTRAP_REPLICATES,seed=RANDOM_SEED):
    data = frame.sort_values("target_date").reset_index(drop=True).copy()
    data["target_date"] = pd.to_datetime(data["target_date"])
    data["block"] = ((data["target_date"].dt.dayofyear-1)//7).astype(int)
    truth = data["truth"].to_numpy(dtype=float)
    pa = data[a].to_numpy(dtype=float)
    pb = data[b].to_numpy(dtype=float)
    groups = {
        block:data.index[data["block"].eq(block)].to_numpy()
        for block in data["block"].unique()
    }
    keys = np.array(list(groups))
    rng = np.random.default_rng(seed)
    maediff = np.empty(reps)
    rmsediff = np.empty(reps)
    for rep in range(reps):
        sample = np.concatenate([
            groups[block]
            for block in rng.choice(keys,len(keys),replace=True)
        ])
        ea = pa[sample]-truth[sample]
        eb = pb[sample]-truth[sample]
        maediff[rep] = np.mean(np.abs(ea))-np.mean(np.abs(eb))
        rmsediff[rep] = np.sqrt(np.mean(ea**2))-np.sqrt(np.mean(eb**2))
    return {
        "mae_diff":np.mean(np.abs(pa-truth))-np.mean(np.abs(pb-truth)),
        "mae_low":np.quantile(maediff,.025),
        "mae_high":np.quantile(maediff,.975),
        "rmse_diff":np.sqrt(np.mean((pa-truth)**2))-np.sqrt(np.mean((pb-truth)**2)),
        "rmse_low":np.quantile(rmsediff,.025),
        "rmse_high":np.quantile(rmsediff,.975)
    }

In [32]:
# Tune transfer predictors
choice_rows = []
for key in active_canada_targets:
    frame = canada_forecast_samples[key]
    train = frame[(frame["year"]<=QUALIFICATION_END_YEAR)&frame["truth"].notna()]
    validation = frame[(frame["year"]==VALIDATION_YEAR)&frame["truth"].notna()]
    ytrain = train["truth"].to_numpy(dtype=float)
    yvalidation = validation["truth"].to_numpy(dtype=float)
    selected = canada_models[key]["taps"]
    Xtrain = forecast_features(key,train,selected)
    Xvalidation = forecast_features(key,validation,selected)
    trials = []
    for index,(depth,leaf,features) in enumerate(product([4,8,None],[2,5],[.5,1.0])):
        forest = Pipeline([
            ("imputer",SimpleImputer(strategy="median",add_indicator=True)),
            ("model",RandomForestRegressor(
                n_estimators=200,
                max_depth=depth,
                min_samples_leaf=leaf,
                max_features=features,
                random_state=RANDOM_SEED,
                n_jobs=-1
            ))
        ])
        forest.fit(Xtrain,ytrain)
        mae = prediction_metrics(forest.predict(Xvalidation),yvalidation)["mae"]
        trials.append((mae,index,depth,leaf,features))
    _,_,depth,leaf,features = min(trials)
    choice_rows.append({
        "target":key,
        "rf_depth":depth,
        "rf_leaf":leaf,
        "rf_features":features
    })
canada_predictor_choices = pd.DataFrame(choice_rows)
display(canada_predictor_choices)

,target,rf_depth,rf_leaf,rf_features
0,BC_Vancouver,4,2,0.5
1,ON_Toronto,4,5,0.5


In [33]:
# Evaluate transfer validation
validation_rows = []
for key in active_canada_targets:
    frame = canada_forecast_samples[key]
    train = frame[(frame["year"]<=QUALIFICATION_END_YEAR)&frame["truth"].notna()]
    validation = frame[(frame["year"]==VALIDATION_YEAR)&frame["truth"].notna()]
    ytrain = train["truth"].to_numpy(dtype=float)
    yvalidation = validation["truth"].to_numpy(dtype=float)
    selected = canada_models[key]["taps"]
    choice = canada_predictor_choices[canada_predictor_choices["target"].eq(key)].iloc[0]
    depth = None if pd.isna(choice["rf_depth"]) else int(choice["rf_depth"])
    leaf = int(choice["rf_leaf"])
    features = float(choice["rf_features"])
    predictions = {"Training mean":np.full(len(yvalidation),ytrain.mean())}
    gtrain,_ = graph_predictions(key,train,selected)
    gvalidation,_ = graph_predictions(key,validation,selected)
    intercept,slope = calibrate(gtrain,ytrain)
    predictions["Graph diffusion"] = intercept+slope*gvalidation
    ptrain = latest_average(key,train,selected)
    pvalidation = latest_average(key,validation,selected)
    fill = np.nanmedian(ptrain)
    intercept,slope = calibrate(np.nan_to_num(ptrain,nan=fill),ytrain)
    predictions["Persistence"] = intercept+slope*np.nan_to_num(pvalidation,nan=fill)
    Xtrain = forecast_features(key,train,selected)
    Xvalidation = forecast_features(key,validation,selected)
    forest = Pipeline([
        ("imputer",SimpleImputer(strategy="median",add_indicator=True)),
        ("model",RandomForestRegressor(
            n_estimators=500,
            max_depth=depth,
            min_samples_leaf=leaf,
            max_features=features,
            random_state=RANDOM_SEED,
            n_jobs=-1
        ))
    ])
    forest.fit(Xtrain,ytrain)
    predictions["Random forest"] = forest.predict(Xvalidation)
    validation_rows.extend({
        "target":key,
        "protocol":canada_eval_protocol[key],
        "model":name,
        **prediction_metrics(prediction,yvalidation)
    } for name,prediction in predictions.items())
canada_validation_results = pd.DataFrame(validation_rows).sort_values(["target","mae"])
display(canada_validation_results.round(6))

,target,protocol,model,n,mae,rmse,bias,correlation
2,BC_Vancouver,full85,Persistence,147,2.756858,4.669787,0.338521,0.302699
3,BC_Vancouver,full85,Random forest,147,2.800767,4.833688,0.271544,0.283390
1,BC_Vancouver,full85,Graph diffusion,147,2.893613,4.696944,0.379098,0.214047
0,BC_Vancouver,full85,Training mean,147,3.381909,4.822695,1.037339,NaN
6,ON_Toronto,full85,Persistence,138,5.878495,11.537991,-3.879744,0.524572
5,ON_Toronto,full85,Graph diffusion,138,5.909521,11.718383,-3.783016,0.405473
7,ON_Toronto,full85,Random forest,138,6.093385,11.989733,-3.993454,0.280497
4,ON_Toronto,full85,Training mean,138,6.375217,12.509401,-4.567699,0.000000


In [34]:
# Evaluate transfer test
test_rows = []
canada_2024_predictions = {}
for key in active_canada_targets:
    frame = canada_forecast_samples[key]
    development = frame[(frame["year"]<=VALIDATION_YEAR)&frame["truth"].notna()]
    test = frame[(frame["year"]==TEST_YEAR)&frame["truth"].notna()]
    ydevelopment = development["truth"].to_numpy(dtype=float)
    ytest = test["truth"].to_numpy(dtype=float)
    selected = canada_models[key]["taps"]
    choice = canada_predictor_choices[canada_predictor_choices["target"].eq(key)].iloc[0]
    depth = None if pd.isna(choice["rf_depth"]) else int(choice["rf_depth"])
    leaf = int(choice["rf_leaf"])
    features = float(choice["rf_features"])
    Xdevelopment = forecast_features(key,development,selected)
    Xtest = forecast_features(key,test,selected)
    forest = Pipeline([
        ("imputer",SimpleImputer(strategy="median",add_indicator=True)),
        ("model",RandomForestRegressor(
            n_estimators=500,
            max_depth=depth,
            min_samples_leaf=leaf,
            max_features=features,
            random_state=RANDOM_SEED,
            n_jobs=-1
        ))
    ])
    forest.fit(Xdevelopment,ydevelopment)
    predictions = {
        "Training mean":np.full(len(ytest),ydevelopment.mean()),
        "Random forest":forest.predict(Xtest)
    }
    gdevelopment,_ = graph_predictions(key,development,selected)
    gtest,gobserved = graph_predictions(key,test,selected)
    intercept,slope = calibrate(gdevelopment,ydevelopment)
    predictions["Graph diffusion"] = intercept+slope*gtest
    pdevelopment = latest_average(key,development,selected)
    ptest = latest_average(key,test,selected)
    fill = np.nanmedian(pdevelopment)
    intercept,slope = calibrate(np.nan_to_num(pdevelopment,nan=fill),ydevelopment)
    predictions["Persistence"] = intercept+slope*np.nan_to_num(ptest,nan=fill)
    test_rows.extend({
        "target":key,
        "protocol":canada_eval_protocol[key],
        "model":name,
        **prediction_metrics(prediction,ytest)
    } for name,prediction in predictions.items())
    frameout = pd.DataFrame({
        "target_date":test["target_date"].to_numpy(),
        "truth":ytest,
        "graph_observed":gobserved
    })
    for name,prediction in predictions.items():
        frameout[name] = prediction
    canada_2024_predictions[key] = frameout
canada_2024_results = pd.DataFrame(test_rows).sort_values(["target","mae"])
canada_2024_audit = pd.DataFrame([
    {
        "target":key,
        "protocol":canada_eval_protocol[key],
        "test_days":len(frame),
        "min_graph_observations":int(frame["graph_observed"].min())
    }
    for key,frame in canada_2024_predictions.items()
])
checks = {
    "Test dates present":canada_2024_audit["test_days"].gt(0).all(),
    "Graph observations present":canada_2024_audit["min_graph_observations"].gt(0).all()
}
canada2024checks = pd.DataFrame({"check":checks.keys(),"passed":checks.values()})
display(canada_2024_audit)
display(canada_2024_results.round(6))
display(canada2024checks)
if not canada2024checks["passed"].all():
    failed = canada2024checks.loc[~canada2024checks["passed"],"check"].tolist()
    raise ValueError("Failed 2024 transfer checks: "+", ".join(failed))

,target,protocol,test_days,min_graph_observations
0,BC_Vancouver,full85,149,26
1,ON_Toronto,full85,126,20


,target,protocol,model,n,mae,rmse,bias,correlation
1,BC_Vancouver,full85,Random forest,149,1.778423,2.295000,0.864406,0.235225
3,BC_Vancouver,full85,Persistence,149,1.833666,2.332927,1.029494,0.290318
2,BC_Vancouver,full85,Graph diffusion,149,1.975035,2.414121,1.183033,0.197590
0,BC_Vancouver,full85,Training mean,149,2.752749,3.095789,2.272970,-0.000000
7,ON_Toronto,full85,Persistence,126,2.802646,3.447943,0.134807,0.377958
6,ON_Toronto,full85,Graph diffusion,126,2.877740,3.538653,0.015336,0.310472
5,ON_Toronto,full85,Random forest,126,2.968351,3.698124,-0.418278,0.151150
4,ON_Toronto,full85,Training mean,126,3.092887,3.717527,0.282862,0.000000


,check,passed
0,Test dates present,True
1,Graph observations present,True


In [35]:
# Evaluate held-out placements
choicemap = canada_predictor_choices.set_index("target").to_dict("index")
placement_rows = []
placement_predictions = []
for key in active_canada_targets:
    frame = canada_forecast_samples[key]
    development = frame[(frame["year"]<=VALIDATION_YEAR)&frame["truth"].notna()]
    test = frame[(frame["year"]==TEST_YEAR)&frame["truth"].notna()]
    ydevelopment = development["truth"].to_numpy(dtype=float)
    ytest = test["truth"].to_numpy(dtype=float)
    choice = choicemap[key]
    depth = None if pd.isna(choice["rf_depth"]) else int(choice["rf_depth"])
    leaf = int(choice["rf_leaf"])
    features = float(choice["rf_features"])
    siteindex = {site:index for index,site in enumerate(canada_models[key]["ids"])}
    rows = canada_selection_results[canada_selection_results["target"].eq(key)]
    for row in rows.itertuples():
        selected = [siteindex[site] for site in str(row.sites).split("|")]
        gdevelopment,_ = graph_predictions(key,development,selected)
        gtest,_ = graph_predictions(key,test,selected)
        intercept,slope = calibrate(gdevelopment,ydevelopment)
        graphprediction = intercept+slope*gtest
        Xdevelopment = forecast_features(key,development,selected)
        Xtest = forecast_features(key,test,selected)
        forest = Pipeline([
            ("imputer",SimpleImputer(strategy="median",add_indicator=True)),
            ("model",RandomForestRegressor(
                n_estimators=500,
                max_depth=depth,
                min_samples_leaf=leaf,
                max_features=features,
                random_state=RANDOM_SEED,
                n_jobs=-1
            ))
        ])
        forest.fit(Xdevelopment,ydevelopment)
        forestprediction = forest.predict(Xtest)
        for predictor,prediction in [
            ("Graph diffusion",graphprediction),
            ("Random forest",forestprediction)
        ]:
            placement_rows.append({
                "target":key,
                "protocol":canada_eval_protocol[key],
                "placement":row.method,
                "predictor":predictor,
                **prediction_metrics(prediction,ytest)
            })
            placement_predictions.extend({
                "target":key,
                "target_date":date,
                "truth":truth,
                "placement":row.method,
                "predictor":predictor,
                "prediction":value
            } for date,truth,value in zip(test["target_date"],ytest,prediction))
canada_2024_placement_results = pd.DataFrame(placement_rows).sort_values(["target","predictor","mae"])
canada_2024_placement_predictions = pd.DataFrame(placement_predictions)
if len(canada_2024_placement_results)!=28:
    raise ValueError("Incomplete deterministic placement evaluation")
display(canada_2024_placement_results.round(6))

,target,protocol,placement,predictor,n,mae,rmse,bias,correlation
6,BC_Vancouver,full85,Weighted degree,Graph diffusion,149,1.912550,2.382130,1.073815,0.193649
4,BC_Vancouver,full85,Target weight,Graph diffusion,149,1.936125,2.405719,1.103807,0.179345
2,BC_Vancouver,full85,Raw response,Graph diffusion,149,1.936717,2.351422,1.072081,0.205435
0,BC_Vancouver,full85,TAPS,Graph diffusion,149,1.975035,2.414121,1.183033,0.197590
8,BC_Vancouver,full85,Geographic coverage,Graph diffusion,149,2.271735,2.738318,1.628813,0.241451
10,BC_Vancouver,full85,D-optimal,Graph diffusion,149,2.346258,2.713993,1.736818,0.199875
12,BC_Vancouver,full85,QR pivoting,Graph diffusion,149,2.760108,3.106101,2.269326,-0.053311
1,BC_Vancouver,full85,TAPS,Random forest,149,1.778423,2.295000,0.864406,0.235225
5,BC_Vancouver,full85,Target weight,Random forest,149,1.837069,2.387534,0.896657,0.206270
3,BC_Vancouver,full85,Raw response,Random forest,149,1.881427,2.362282,0.943762,0.198677


In [36]:
# Bootstrap transfer models
rows = []
pairs = [
    ("Random forest","Graph diffusion"),
    ("Graph diffusion","Persistence"),
    ("Random forest","Persistence")
]
for targetindex,key in enumerate(active_canada_targets):
    predictions = canada_2024_predictions[key]
    for pairindex,(a,b) in enumerate(pairs):
        rows.append({
            "target":key,
            "model_a":a,
            "model_b":b,
            **bootdiff(
                predictions,
                a,
                b,
                BOOTSTRAP_REPLICATES,
                RANDOM_SEED+10*targetindex+pairindex
            )
        })
canada_model_bootstrap = pd.DataFrame(rows)
if len(canada_model_bootstrap)!=6:
    raise ValueError("Incomplete model bootstrap")
display(canada_model_bootstrap.round(6))

,target,model_a,model_b,mae_diff,mae_low,mae_high,rmse_diff,rmse_low,rmse_high
0,BC_Vancouver,Random forest,Graph diffusion,-0.196611,-0.296709,-0.103462,-0.119121,-0.249467,0.006724
1,BC_Vancouver,Graph diffusion,Persistence,0.141369,0.054105,0.230559,0.081194,-0.031553,0.195703
2,BC_Vancouver,Random forest,Persistence,-0.055242,-0.113009,0.009711,-0.037928,-0.111610,0.046054
3,ON_Toronto,Random forest,Graph diffusion,0.090611,-0.180736,0.414648,0.159471,-0.117214,0.472695
4,ON_Toronto,Graph diffusion,Persistence,0.075094,-0.041046,0.203677,0.090710,-0.007152,0.190354
5,ON_Toronto,Random forest,Persistence,0.165705,-0.104867,0.473759,0.250181,-0.026398,0.570948


In [37]:
# Bootstrap placement effects
rows = []
for targetindex,key in enumerate(active_canada_targets):
    for predictorindex,predictor in enumerate(["Graph diffusion","Random forest"]):
        subset = canada_2024_placement_predictions[
            canada_2024_placement_predictions["target"].eq(key)
            &canada_2024_placement_predictions["predictor"].eq(predictor)
            &canada_2024_placement_predictions["placement"].isin(["TAPS","Target weight"])
        ]
        wide = subset.pivot(
            index=["target_date","truth"],
            columns="placement",
            values="prediction"
        ).reset_index()
        rows.append({
            "target":key,
            "predictor":predictor,
            "placement_a":"TAPS",
            "placement_b":"Target weight",
            **bootdiff(
                wide,
                "TAPS",
                "Target weight",
                BOOTSTRAP_REPLICATES,
                100+RANDOM_SEED+10*targetindex+predictorindex
            )
        })
canada_placement_bootstrap = pd.DataFrame(rows)
if len(canada_placement_bootstrap)!=4:
    raise ValueError("Incomplete placement bootstrap")
display(canada_placement_bootstrap.round(6))

,target,predictor,placement_a,placement_b,mae_diff,mae_low,mae_high,rmse_diff,rmse_low,rmse_high
0,BC_Vancouver,Graph diffusion,TAPS,Target weight,0.038910,-0.009573,0.089478,0.008402,-0.058047,0.077245
1,BC_Vancouver,Random forest,TAPS,Target weight,-0.058646,-0.131138,0.005372,-0.092535,-0.174851,-0.002259
2,ON_Toronto,Graph diffusion,TAPS,Target weight,-0.009091,-0.049307,0.031525,-0.004574,-0.050838,0.037877
3,ON_Toronto,Random forest,TAPS,Target weight,0.060312,-0.086185,0.212708,0.137001,-0.034723,0.301939


In [38]:
# Benchmark held-out random placements
random_rows = []
for key in active_canada_targets:
    frame = canada_forecast_samples[key]
    development = frame[(frame["year"]<=VALIDATION_YEAR)&frame["truth"].notna()]
    test = frame[(frame["year"]==TEST_YEAR)&frame["truth"].notna()]
    ydevelopment = development["truth"].to_numpy(dtype=float)
    ytest = test["truth"].to_numpy(dtype=float)
    model = canada_models[key]
    n = len(model["ids"])
    if comb(n,SENSOR_BUDGET)<RANDOM_SETS:
        raise ValueError(f"{key}: fewer than {RANDOM_SETS} distinct sensor sets at budget {SENSOR_BUDGET}")
    rng = np.random.default_rng(RANDOM_SEED)
    seen = set()
    while len(seen)<RANDOM_SETS:
        seen.add(tuple(sorted(rng.choice(n,SENSOR_BUDGET,replace=False).tolist())))
    for draw,selected in enumerate(sorted(seen)):
        selected = list(selected)
        gdevelopment,_ = graph_predictions(key,development,selected)
        gtest,_ = graph_predictions(key,test,selected)
        intercept,slope = calibrate(gdevelopment,ydevelopment)
        random_rows.append({
            "target":key,
            "draw":draw,
            "sites":"|".join(np.asarray(model["ids"])[selected]),
            **prediction_metrics(intercept+slope*gtest,ytest)
        })
canada_random_2024 = pd.DataFrame(random_rows)
canada_random_2024_sets = canada_random_2024[["target","draw","sites"]].copy()
summary = []
for key,subset in canada_random_2024.groupby("target"):
    taps = float(canada_2024_placement_results[
        canada_2024_placement_results["target"].eq(key)
        &canada_2024_placement_results["predictor"].eq("Graph diffusion")
        &canada_2024_placement_results["placement"].eq("TAPS")
    ]["mae"].iloc[0])
    summary.append({
        "target":key,
        "taps_graph_mae":taps,
        "random_mean_mae":subset["mae"].mean(),
        "random_median_mae":subset["mae"].median(),
        "random_p05_mae":subset["mae"].quantile(.05),
        "random_p95_mae":subset["mae"].quantile(.95),
        "taps_better_than_random_fraction":float((subset["mae"]>taps).mean())
    })
canada_random_2024_summary = pd.DataFrame(summary)
checks = {
    "Held-out random sets complete":canada_random_2024.groupby("target").size().eq(RANDOM_SETS).all(),
    "Held-out random sets unique":canada_random_2024_sets.groupby("target")["sites"].nunique().eq(RANDOM_SETS).all()
}
random2024checks = pd.DataFrame({"check":checks.keys(),"passed":checks.values()})
display(canada_random_2024_summary.round(6))
display(random2024checks)
if not random2024checks["passed"].all():
    raise ValueError("Held-out random-set audit failed")

,target,taps_graph_mae,random_mean_mae,random_median_mae,random_p05_mae,random_p95_mae,taps_better_than_random_fraction
0,BC_Vancouver,1.975035,1.953396,1.948718,1.832157,2.058863,0.36
1,ON_Toronto,2.877740,2.796198,2.793743,2.685151,2.932124,0.13


,check,passed
0,Held-out random sets complete,True
1,Held-out random sets unique,True


In [39]:
# Freeze final transfer results
bestmodel = canada_2024_results.sort_values(["target","mae"]).groupby(
    "target",as_index=False
).first()[["target","protocol","model","mae","rmse","correlation"]]
bestmodel = bestmodel.rename(columns={
    "model":"best_model",
    "mae":"best_model_mae",
    "rmse":"best_model_rmse",
    "correlation":"best_model_correlation"
})
bestgraph = canada_2024_placement_results[
    canada_2024_placement_results["predictor"].eq("Graph diffusion")
].sort_values(["target","mae"]).groupby(
    "target",as_index=False
).first()[["target","placement","mae"]]
bestgraph = bestgraph.rename(columns={
    "placement":"best_graph_placement",
    "mae":"best_graph_mae"
})
bestrf = canada_2024_placement_results[
    canada_2024_placement_results["predictor"].eq("Random forest")
].sort_values(["target","mae"]).groupby(
    "target",as_index=False
).first()[["target","placement","mae"]]
bestrf = bestrf.rename(columns={
    "placement":"best_rf_placement",
    "mae":"best_rf_mae"
})

modelbase = canada_taps_results[[
    "target","K","target_effective_dimension","sensor_budget",
    "span_error","risk","rank","full_state_risk",
    "exact_at_fixed_budget","taps_first_exact_budget","rank_at_taps_first_exact",
    "taps_first_full_rank_budget","target_state_separation_at_taps_first_exact"
]].copy()

final_transfer = modelbase.merge(
    bestmodel,on="target"
).merge(
    bestgraph,on="target"
).merge(
    bestrf,on="target"
).merge(
    canada_random_2024_summary[["target","taps_better_than_random_fraction"]],
    on="target"
)

final_transfer["full_state_identifiable_at_fixed_budget"] = (
    final_transfer["rank"]>=final_transfer["K"]
)

checks = {
    "Two active targets":len(final_transfer)==2,
    "Fixed-budget results finite":np.isfinite(
        final_transfer[["span_error","risk","best_model_mae","best_graph_mae","best_rf_mae"]].to_numpy()
    ).all(),
    "Test-year results present":final_transfer["best_model"].notna().all(),
    "Exactness outcome recorded":final_transfer["exact_at_fixed_budget"].notna().all(),
    "TAPS exact-budget scan recorded":final_transfer["taps_first_exact_budget"].notna().all()
}
finalchecks = pd.DataFrame({"check":checks.keys(),"passed":checks.values()})

display(final_transfer.round(6))
display(finalchecks)

if not finalchecks["passed"].all():
    failed = finalchecks.loc[~finalchecks["passed"],"check"].tolist()
    raise ValueError("Failed final transfer checks: "+", ".join(failed))

,target,K,target_effective_dimension,sensor_budget,span_error,risk,rank,full_state_risk,exact_at_fixed_budget,taps_first_exact_budget,...,best_model,best_model_mae,best_model_rmse,best_model_correlation,best_graph_placement,best_graph_mae,best_rf_placement,best_rf_mae,taps_better_than_random_fraction,full_state_identifiable_at_fixed_budget
0,BC_Vancouver,23,9,10,0.000000,0.158076,19,1.0,True,4,...,Random forest,1.778423,2.295000,0.235225,Weighted degree,1.91255,TAPS,1.778423,0.36,False
1,ON_Toronto,34,13,10,0.000091,0.167137,30,1.0,False,11,...,Persistence,2.802646,3.447943,0.377958,TAPS,2.87774,D-optimal,2.609277,0.13,False


,check,passed
0,Two active targets,True
1,Fixed-budget results finite,True
2,Test-year results present,True
3,Exactness outcome recorded,True
4,TAPS exact-budget scan recorded,True


In [40]:
# Build transfer export tables
source_manifest = pd.DataFrame([
    ["EPA AirData","https://aqs.epa.gov/aqsweb/airdata/download_files.html","Oregon and Washington feasibility",True,ACCESS_DATE_UTC,"Accessed during this notebook run."],
    ["Defra UK-AIR AURN","https://uk-air.defra.gov.uk/","England feasibility",True,ACCESS_DATE_UTC,"Accessed during this notebook run."],
    ["ECCC NAPS PM2.5","https://data-donnees.az.ec.gc.ca/","2018-2024 hourly PM2.5 with coordinates",True,ACCESS_DATE_UTC,"Accessed during this notebook run."],
    ["Statistics Canada census tracts","https://www12.statcan.gc.ca/census-recensement/2021/geo/sip-pis/","2021 census-tract boundaries",True,ACCESS_DATE_UTC,"Accessed during this notebook run."],
    ["Statistics Canada 98-10-0014","https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?pid=9810001401","2021 population counts",True,ACCESS_DATE_UTC,"Accessed during this notebook run."],
    ["EEA prior audit","https://www.eea.europa.eu/en/datahub/datahubitem-view/778ef9f5-6293-4846-badd-56a29c70880d","Portugal feasibility; Spain and Greece retrieval unresolved",False,pd.NA,"Prior audit; exact access date was not recorded, so none is invented here."]
],columns=["source","url","use","reproduced_here","access_date_utc","provenance_note"])
tables = {
    "transfer_feasibility.csv":transfer_feasibility,
    "prior_europe_audit.csv":prior_europe_audit,
    "naps_download_audit.csv":naps_download_audit,
    "naps_value_audit.csv":naps_value_audit,
    "canada_province_audit.csv":feasibilityprovinceaudit,
    "canada_candidate_audit.csv":candidateprovinceaudit,
    "canada_station_coverage.csv":canada_feasibility_stats,
    "canada_candidate_coverage.csv":canada_candidate_stats,
    "canada_station_years.csv":canada_station_years,
    "canada_region_audit.csv":canada_region_audit,
    "canada_graph_audit.csv":canada_graph_audit,
    "canada_spectral_audit.csv":canada_spectral_audit,
    "canada_census_audit.csv":canada_census_audit,
    "canada_target_audit.csv":canada_target_audit,
    "canada_target_weights.csv":canada_target_weight_table,
    "canada_taps_results.csv":canada_taps_results,
    "canada_taps_exactness_scan.csv":canada_taps_exactness_scan,
    "canada_model_based_placements.csv":canada_placement_results,
    "canada_model_random.csv":canada_random_model,
    "canada_model_random_sets.csv":canada_model_random_sets,
    "canada_model_random_summary.csv":random_model_summary,
    "canada_selected_sites.csv":canada_selection_results,
    "canada_forecast_audit.csv":canada_forecast_audit,
    "canada_predictor_choices.csv":canada_predictor_choices,
    "canada_validation_results.csv":canada_validation_results,
    "canada_2024_results.csv":canada_2024_results,
    "canada_2024_placement_results.csv":canada_2024_placement_results,
    "canada_2024_placement_predictions.csv":canada_2024_placement_predictions,
    "canada_model_bootstrap.csv":canada_model_bootstrap,
    "canada_placement_bootstrap.csv":canada_placement_bootstrap,
    "canada_random_2024.csv":canada_random_2024,
    "canada_random_2024_sets.csv":canada_random_2024_sets,
    "canada_random_2024_summary.csv":canada_random_2024_summary,
    "transfer_final_summary.csv":final_transfer,
    "source_manifest.csv":source_manifest,
    "washington_coverage.csv":us_audits["Washington"]["stats"],
    "washington_years.csv":us_audits["Washington"]["years"],
    "washington_poc.csv":us_audits["Washington"]["poc"],
    "washington_coordinates.csv":us_audits["Washington"]["coords"],
    "oregon_coverage.csv":us_audits["Oregon"]["stats"],
    "oregon_years.csv":us_audits["Oregon"]["years"],
    "oregon_poc.csv":us_audits["Oregon"]["poc"],
    "oregon_coordinates.csv":us_audits["Oregon"]["coords"],
    "uk_aurn_metadata.csv":ukmeta,
    "england_files.csv":englandfiles,
    "england_coverage.csv":englandstats,
    "england_years.csv":englandyears
}
manifest = {
    "notebook":"06-transfer.ipynb",
    "purpose":"Geographic transfer feasibility, prospective Canadian TAPS replication, and held-out prediction",
    "run_started_utc":RUN_STARTED_UTC,
    "access_date_utc":ACCESS_DATE_UTC,
    "feasibility_audit_years":[2018,2024],
    "feasibility_minimum_years":5,
    "candidate_qualification_years":[QUALIFICATION_START_YEAR,QUALIFICATION_END_YEAR],
    "minimum_qualification_years":MIN_QUALIFICATION_YEARS,
    "minimum_season_days":MIN_SEASON_DAYS,
    "graph_training_years":[QUALIFICATION_START_YEAR,QUALIFICATION_END_YEAR],
    "graph_validation_year":VALIDATION_YEAR,
    "graph_robust_edge_ratio":GRAPH_PRUNE_RATIO,
    "test_year":TEST_YEAR,
    "target_coverage_threshold":TARGET_COVERAGE_THRESHOLD,
    "targets":active_canada_targets,
    "sensor_budget":SENSOR_BUDGET,
    "exact_tolerance":EXACT_TOLERANCE,
    "random_sets":RANDOM_SETS,
    "bootstrap_replicates":BOOTSTRAP_REPLICATES,
    "seed":RANDOM_SEED,
    "observation_offsets_days":OBSERVATION_OFFSETS,
    "prior_unreproduced_rows":["Portugal","Spain","Greece"]
}

In [41]:
# Export transfer archive
output = Path("transfer_results")
shutil.rmtree(output,ignore_errors=True)
output.mkdir()
for name,table in tables.items():
    table.to_csv(output/name,index=False)
for province,tuning in canada_graph_tuning.items():
    tuning.to_csv(output/f"{province.lower()}_graph_tuning.csv",index=False)
for key,frame in canada_2024_predictions.items():
    frame.to_csv(output/f"{key.lower()}_2024_predictions.csv",index=False)
with open(output/"manifest.json","w") as file:
    json.dump(manifest,file,indent=2)
required = [
    "transfer_feasibility.csv",
    "prior_europe_audit.csv",
    "canada_station_years.csv",
    "canada_candidate_audit.csv",
    "canada_candidate_coverage.csv",
    "canada_graph_audit.csv",
    "canada_target_audit.csv",
    "canada_taps_results.csv",
    "canada_taps_exactness_scan.csv",
    "canada_model_random_sets.csv",
    "canada_2024_results.csv",
    "canada_2024_placement_results.csv",
    "canada_random_2024.csv",
    "canada_random_2024_sets.csv",
    "transfer_final_summary.csv",
    "source_manifest.csv",
    "manifest.json"
]
missing = [name for name in required if not (output/name).exists()]
if missing:
    raise FileNotFoundError("Missing transfer exports: "+", ".join(missing))
archive = Path("transfer.zip")
if archive.exists():
    archive.unlink()
with zipfile.ZipFile(archive,"w",compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(output.iterdir()):
        bundle.write(path,arcname=path.name)
print("Exported files:",len(list(output.iterdir())))
print("Archive:",archive)

Exported files: 52
Archive: transfer.zip


In [42]:
# Audit transfer archive
with zipfile.ZipFile(archive) as bundle:
    corrupt = bundle.testzip()
    members = set(bundle.namelist())
    stationyears = pd.read_csv(bundle.open("canada_station_years.csv"),dtype={"site":str})
    feasibilityaudit = pd.read_csv(bundle.open("canada_province_audit.csv"))
    candidateaudit = pd.read_csv(bundle.open("canada_candidate_audit.csv"))
    randommodelsets = pd.read_csv(bundle.open("canada_model_random_sets.csv"),dtype={"sites":str})
    randomtestsets = pd.read_csv(bundle.open("canada_random_2024_sets.csv"),dtype={"sites":str})
    sources = pd.read_csv(bundle.open("source_manifest.csv"))

archivechecks = {
    "Archive readable":corrupt is None,
    "Required files present":set(required).issubset(members),
    "Station-year schema correct":set(stationyears.columns)=={"site","province","year","days"},
    "Station years span 2018-2024":stationyears["year"].min()==2018 and stationyears["year"].max()==2024,
    "Feasibility and candidate audits both present":len(feasibilityaudit)>0 and len(candidateaudit)>0,
    "Prospective Ontario candidate count differs transparently":int(candidateaudit.loc[candidateaudit["province"].eq("ON"),"qualified_sites"].iloc[0])>=SENSOR_BUDGET,
    "Model random sets complete":randommodelsets.groupby("target")["sites"].nunique().eq(RANDOM_SETS).all(),
    "Held-out random sets complete":randomtestsets.groupby("target")["sites"].nunique().eq(RANDOM_SETS).all(),
    "Current sources have access dates":sources.loc[sources["reproduced_here"],"access_date_utc"].notna().all(),
    "Prior EEA date not invented":sources.loc[~sources["reproduced_here"],"access_date_utc"].isna().all()
}
archivechecks = pd.DataFrame({"check":archivechecks.keys(),"passed":archivechecks.values()})
display(archivechecks)
if not archivechecks["passed"].all():
    failed = archivechecks.loc[~archivechecks["passed"],"check"].tolist()
    raise ValueError("Failed transfer archive checks: "+", ".join(failed))

,check,passed
0,Archive readable,True
1,Required files present,True
2,Station-year schema correct,True
3,Station years span 2018-2024,True
4,Feasibility and candidate audits both present,True
5,Prospective Ontario candidate count differs tr...,True
6,Model random sets complete,True
7,Held-out random sets complete,True
8,Current sources have access dates,True
9,Prior EEA date not invented,True


In [43]:
# Download transfer archive
files.download(str(archive))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>